# Phase 4 - Court Judgment Prediction (CJPE) - Architecture 1 (LLaMA-style decoder)

**Task:** binary court-judgment-outcome prediction (labels 0 / 1) from the full text of a case. Primary metric: **macro-F1**.
**Phase-1 model:** Architecture 1 (LLaMA-style decoder) - RMSNorm + RoPE + SwiGLU decoder (the modified variant adds GQA, QK-norm, tied embeddings, value residual, MTP and a source embedding). The source embedding is set to *Supreme Court* because every CJPE case is a Supreme Court judgment.
**Adaptation (this notebook):** **LoRMA (low-rank multiplicative adaptation - the compulsory method)**, one `METHOD = ...` switch away from the other Assignment-4 techniques (LoRMA, LoRA, adapters, soft prompts) and full fine-tuning.
**No knowledge distillation** in this notebook (distillation is done in `Phase4_Arch_1_LoRMA_KD.ipynb`).

**How a prediction is made.** A judgment is cut into overlapping **chunks** of `CHUNK_LEN` tokens (stride `CHUNK_STRIDE`); very long cases keep their head and their tail. The Phase-1 model reads every chunk - causal decoder: the **mean** of the token states and the state of the **last** token of a chunk are the chunk vector. A **hierarchical document head** adds an embedding of the chunk's relative position in the judgment (section-aware), pools the chunk vectors (`DOC_POOL`: attention / mean / BiGRU / last chunk) and a small MLP gives the outcome. `SECTION_MODE` selects which part of the judgment is read (`full`, `last`, `first`, `head_tail`).

| step | what happens |
|---|---|
| 1-3 | setup, configuration, your Phase-1 tokenizer (unchanged) |
| 4-5 | load CJPE data, tokenise, class balance, chunking and batching |
| 6 | macro-F1 / precision / recall, threshold tuning, calibration, false-positive / false-negative analysis |
| 7-9 | your Phase-1 model (unchanged), adaptation methods of Assignment 4, document classifier |
| 10-12 | training code, then run: train, validate, section analysis, predict test, write `submission.zip` |
| 13 | save trainable weights and `results.json` |

**Before running:** attach your Phase-1 checkpoint (`best.pt`) and `kratikasum/tokenizer` as Kaggle inputs and keep the secret `HF`. Run once with `SMOKE_TEST = True`, then switch it off. Only Phase-1 weights and the provided CJPE data are used.

**Outputs** (in `/kaggle/working/<RUN_NAME>/`): `prediction.json` (`[{"id": ..., "pred": 0/1}, ...]`), `submission.zip`, `results.json`, `dev_probs.npy` / `test_probs.npy` (for ensembling), the trained weights.

> **v2 changes (after a first run that stayed at chance: loss ~0.69, dev macro-F1 ~0.57):**
> lower backbone LR (1e-4) and head LR (5e-4), longer warm-up, 6 chunks per training document, 12 000 documents per epoch x 2 epochs,
> quick-dev set 800 documents, patience 12 and no early stop before 30% of the run, features from layers 8 / 12 / 16 instead of the last layer only,
> duplicate / leakage check, TF-IDF reference model, overfit sanity test (before 12a) and a dev-tuned blend with TF-IDF (12g).

## 1. Setup

### Imports, Hugging Face login, GPU check

In [51]:
# Import the standard Python tools we need
import os, re, sys, json, glob, math, time, random, zipfile, gc, pickle, copy  # os/sys = files & system, json = read/write JSON, glob = find files, math = maths, time = timers, random = random numbers, zipfile = make zip, pickle = load saved objects
from collections import Counter  # Counter counts how many times each item appears
from dataclasses import dataclass, asdict, fields  # dataclass = easy "settings" classes
from typing import List, Dict, Tuple, Optional  # type hints (only for readability, they do not change behaviour)
import numpy as np  # numpy = fast arrays and maths
import torch  # PyTorch = the deep learning library
import torch.nn as nn  # nn = neural network building blocks (layers)
import torch.nn.functional as F  # F = functions like softmax, cross_entropy
from huggingface_hub import login, hf_hub_download  # tools to log in to Hugging Face and download files
from tqdm.auto import tqdm  # tqdm = progress bars

try:                                                     # Kaggle secret 'HF' (same as in your Phase-1 / 2 / 3 notebooks)
    from kaggle_secrets import UserSecretsClient  # Kaggle's tool for reading private secrets
    HF_TOKEN = UserSecretsClient().get_secret("HF")  # read our Hugging Face token from the secret called "HF"
    os.environ["HF_TOKEN"] = HF_TOKEN  # store it as an environment variable so other tools can find it
    login(token=HF_TOKEN)  # log in to Hugging Face using the token
except Exception as ex:  # if anything above fails (e.g. not running on Kaggle)...
    HF_TOKEN = os.environ.get("HF_TOKEN")  # ...try to read the token from the environment instead
    print(f"Kaggle secret 'HF' not available ({type(ex).__name__}); using env HF_TOKEN / local files")  # tell the user what happened

print('CUDA available:', torch.cuda.is_available())  # show whether a GPU can be used
for i in range(torch.cuda.device_count()):  # loop over every GPU found
    print(f'GPU {i}:', torch.cuda.get_device_name(i))  # print the name of each GPU
N_GPU = torch.cuda.device_count()  # remember how many GPUs we have
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # use the GPU if there is one, otherwise the CPU
use_bf16 = device == 'cuda' and torch.cuda.get_device_capability()[0] >= 8      # T4 / P100: fp16 + GradScaler, Ampere+: bf16
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16  # pick the low-precision number type used to speed up training

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


CUDA available: True
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Configuration
Every setting of the experiment is in this section.

### Paths and run name

In [52]:
# ---------- where things are ----------
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'  # name of the Hugging Face dataset that holds the data
PHASE4_DIR      = 'Phase_4_CJPE'        # Train/train.json, Train/dev.json, Test/test.json inside that dataset
LOCAL_DATA_DIR  = None                  # or e.g. '/kaggle/input/<dataset>/Phase_4_CJPE'
PHASE1_CKPT     = ''                    # your Phase-1 best.pt / last.pt   ('' = search /kaggle/input)
CKPT_HINT       = 'arch1'       # used by that search: the path must contain this text ('-' and '_' are ignored)
TOKENIZER_PKL   = ''                    # '' = search /kaggle/input, then download kratikasum/tokenizer
RUN_NAME        = 'p4_arch1_lorma_v2'  # a name for this experiment (also used as the output folder name)
OUT_ROOT        = '/kaggle/working' if os.path.isdir('/kaggle/working') else './outputs'  # save results in Kaggle's working folder, or a local "outputs" folder

### Long-document representation, pooling and class balance

In [53]:
# ---------- how a long judgment is represented ----------
SEED            = 42  # fixed random seed so results can be repeated
CHUNK_LEN       = 1024                  # tokens per chunk: the Phase-1 model reads one chunk at a time
CHUNK_STRIDE    = 896                   # < CHUNK_LEN -> neighbouring chunks overlap by CHUNK_LEN - CHUNK_STRIDE tokens
MAX_DOC_TOKENS  = 12288                 # a longer judgment keeps its first HEAD_FRAC and its last (1 - HEAD_FRAC) share of tokens
HEAD_FRAC       = 0.25  # share of the kept tokens that come from the START of a long document (the rest come from the END)
SECTION_MODE    = 'full'                # which part of the judgment is read: 'full' | 'last' | 'first' | 'head_tail'
TRAIN_CHUNKS    = 6                     # chunks per document in one training step (random subset, document order is kept)
TRAIN_KEEP_LAST = True                  # the last chunk (the court's closing reasoning) is always one of them
MAX_EVAL_CHUNKS = 24                    # chunks per document at validation / test time (evenly spread, first + last included)
DOC_POOL        = 'attn'                # chunk vectors -> document vector: 'attn' | 'mean' | 'gru' | 'last'
HEAD_HIDDEN     = 256                   # width of the document-level head
N_POS_BINS      = 8                     # the relative position of a chunk inside the judgment is embedded (section-aware head)
CW_POWER        = 0.5                   # class weight = (N / (C * count_label)) ** CW_POWER   (0 = off)
LABEL_SMOOTH    = 0.05  # label smoothing: makes the model slightly less over-confident

### Adaptation method

In [54]:
# ---------- adaptation method (the four techniques of Assignment 4, plus full fine-tuning as a reference) ----------
METHOD          = 'lorma'            # 'lorma' | 'lora' | 'adapter' | 'soft_prompt' | 'full'   (see section 8)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05              # 'lora' / 'lorma': rank, alpha (scale = alpha / r = 2, as in Assignment 4), dropout
PEFT_TARGETS    = ('q', 'k', 'v', 'o', 'gate', 'up', 'down')   # linear layers that receive LoRA / LoRMA matrices
ADAPTER_BOTTLENECK = 64                 # 'adapter': bottleneck width of the residual adapter after every block
N_PROMPT        = 32                    # 'soft_prompt': number of learned input vectors
PROMPT_INIT_TEXT = ' court judgment prediction: read the case and decide whether the appeal is accepted or rejected:'  # text whose embeddings start the soft prompt
BACKBONE_DROPOUT = 0.1                  # dropout inside the Phase-1 blocks while fine-tuning

### Training

In [55]:
# ---------- training ----------
EPOCHS          = 2  # passes over the (sub-sampled) training set
LR_BY_METHOD    = dict(lorma=1e-4, lora=1e-4, adapter=5e-4, soft_prompt=3e-3, full=2e-5)   # lowered: 3e-4 on all 7 linears was too aggressive
HEAD_LR         = 5e-4                  # document head (always trained from scratch)
BATCH_DOCS, GRAD_ACCUM = 2, 4           # documents per forward pass x accumulation  (= 8 documents per optimizer step)
EPOCH_DOCS      = 12000                 # documents per epoch (1500 steps); a different random subset each epoch
GRAD_CKPT       = True                  # saves GPU memory (~30% slower). Set False if you have memory to spare
EVAL_EVERY      = 250                   # optimizer steps between two quick validations
QUICK_DEV_DOCS, QUICK_EVAL_CHUNKS = 800, 8    # bigger quick-dev set -> far less noisy model selection
EVAL_CHUNKS_PER_BATCH = 8               # chunks per forward pass at inference time
LOG_EVERY       = 20  # print/record the training loss every 20 steps
PATIENCE        = 12                    # stop after this many quick validations without improvement (never before 30% of the run)
MAX_TRAIN_MINUTES = 300                 # wall-clock cap: the LR schedule is stretched so that it always finishes decaying
APPLY_TUNED_THRESHOLD = True            # test predictions use the decision threshold that maximises dev macro-F1 (else 0.5)

### Quick dry run, seeds and output folder

In [56]:
# ---------- quick dry run ----------
SMOKE_TEST = False                      # True = a handful of documents and 6 steps: checks the whole pipeline in minutes
MAX_DOCS = MAX_STEPS = None  # None means "no limit" on documents / steps
if SMOKE_TEST:  # only when doing the quick test run
    MAX_DOCS, MAX_STEPS = 24, 6  # use just 24 documents and stop after 6 steps
    RUN_NAME += '_smoke'  # add "_smoke" to the run name so real results are not overwritten
    EPOCHS, EVAL_EVERY, QUICK_DEV_DOCS, MAX_EVAL_CHUNKS = 1, 3, 12, 6  # shrink all the other settings too

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # fix the random seed in Python, numpy and PyTorch
LR = LR_BY_METHOD[METHOD]  # pick the learning rate that matches the chosen method
OUTPUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)  # full path of the folder where this run's files go
os.makedirs(OUTPUT_DIR, exist_ok=True)  # create that folder (no error if it already exists)
print('run:', RUN_NAME, '| method:', METHOD, '| output folder:', OUTPUT_DIR)  # show the main settings

run: p4_arch1_lorma_v2 | method: lorma | output folder: /kaggle/working/p4_arch1_lorma_v2


## 3. Your Phase-1 tokenizer
The class is your Phase-1 code, unchanged.

### Tokenizer class

In [57]:
class ByteLevelBPE:  # our tokenizer: turns text into numbers (token ids) and back
    """Your tokenizer class (training code removed: we load the already-trained pickle)."""
    def __init__(self):  # runs when a new tokenizer object is created
        self.vocab: Dict[bytes, int] = {}  # dictionary: token (bytes) -> id number
        self.inverse_vocab: Dict[int, bytes] = {}  # dictionary: id number -> token (bytes)
        self.merges: Dict[Tuple[bytes, bytes], Tuple[bytes, int]] = {}  # merge rules: which two tokens join into one, and their priority
        self.word_end_marker = b'</w>'  # special mark added at the end of every word
        self.reserved_tokens = [b'<pad>', b'<unk>', b'<bos>', b'<eos>']  # special tokens: padding, unknown, begin, end

    def tokenize(self, text: str) -> List[bytes]:  # text -> list of tokens (as bytes)
        words = text.split()  # split the text into words at spaces
        tokens = []  # the final list of tokens
        for w in words:  # handle one word at a time
            word_bytes = w.encode('utf-8')  # turn the word into raw bytes
            split = [bytes([b]) for b in word_bytes] + [self.word_end_marker]  # start with one token per byte, plus the end-of-word mark
            while True:  # keep merging until no more merges are possible
                best_pair, best_rank = None, float('inf')  # remember the best pair to merge (lowest rank = highest priority)
                for i in range(len(split) - 1):  # look at every neighbouring pair
                    pair = (split[i], split[i+1])  # the pair of two neighbouring tokens
                    if pair in self.merges:  # is there a rule to merge this pair?
                        rank = self.merges[pair][1]  # get the priority number of that rule
                        if rank < best_rank:  # is it better than the best so far?
                            best_rank, best_pair = rank, pair  # yes -> remember it
                if best_pair is None:  # no mergeable pair left
                    break  # so this word is finished
                new_split, i = [], 0  # build the new list after merging
                while i < len(split):  # walk through the current tokens
                    if i < len(split) - 1 and split[i] == best_pair[0] and split[i+1] == best_pair[1]:  # found the pair to merge
                        new_split.append(self.merges[best_pair][0])  # add the merged token
                        i += 2  # skip both old tokens
                    else:  # not the pair
                        new_split.append(split[i])  # copy the token unchanged
                        i += 1  # move to the next token
                split = new_split  # use the merged list for the next round
            tokens.extend(split)  # add this word's tokens to the result
        return tokens  # give back all tokens

    def encode(self, text: str) -> List[int]:  # text -> list of integer ids
        unk_id = self.vocab[b'<unk>']  # id used for tokens we do not know
        return [self.vocab.get(token, unk_id) for token in self.tokenize(text)]  # look up the id of every token

    def decode(self, token_ids: List[int]) -> str:  # list of ids -> text
        skip_ids = {self.vocab[t] for t in self.reserved_tokens if t in self.vocab}  # ids of special tokens (not shown in the text)
        bytes_list = []  # collect the bytes of every token
        for tid in token_ids:  # go through every id
            if tid in skip_ids:  # special token?
                continue  # skip it
            bytes_list.append(self.inverse_vocab.get(tid, b''))  # look up the token bytes for this id
        joined = b''.join(bytes_list)  # glue all the bytes together
        return joined.replace(self.word_end_marker, b' ').decode('utf-8', errors='replace').strip()  # end-of-word marks become spaces, then bytes -> text

    @classmethod
    def load(cls, filepath: str):  # build a tokenizer from a saved file
        with open(filepath, 'rb') as f:  # open the pickle file for reading
            data = pickle.load(f)  # read the saved vocab and merge rules
        bpe = cls()  # make an empty tokenizer
        bpe.vocab = data['vocab']  # fill in the vocabulary
        bpe.inverse_vocab = {v: k for k, v in bpe.vocab.items()}  # build the reverse dictionary (id -> token)
        bpe.merges = data['merges']  # fill in the merge rules
        return bpe  # return the ready tokenizer

    def __len__(self):  # makes len(tokenizer) work
        return len(self.vocab)  # number of tokens in the vocabulary

    @property
    def eos_token_id(self):  # shortcut to get the id of the <eos> token
        return self.vocab.get(b'<eos>')  # look it up in the vocabulary

### Load the trained tokenizer, check vocabulary size

In [58]:
if TOKENIZER_PKL and os.path.exists(TOKENIZER_PKL):  # if the user gave a valid tokenizer file path
    tok_path = TOKENIZER_PKL  # use it
else:  # otherwise look for it
    cands = glob.glob('/kaggle/input/**/bpe_tokenized_txt.pkl', recursive=True)  # search every Kaggle input folder for the tokenizer file
    if cands:  # found at least one
        tok_path = cands[0]  # use the first one
    else:                                   # dataset not attached as input -> download it
        import kagglehub  # tool to download Kaggle datasets
        tok_path = os.path.join(kagglehub.dataset_download('kratikasum/tokenizer'), 'bpe_tokenized_txt.pkl')  # download and point to the file
tokenizer = ByteLevelBPE.load(tok_path)  # load the tokenizer from the file

VOCAB_SIZE = max(tokenizer.vocab.values()) + 1  # number of embedding rows needed = biggest id + 1
PAD_ID, EOS = tokenizer.vocab[b'<pad>'], tokenizer.eos_token_id  # ids of the padding token and the end-of-text token
print(f'tokenizer: {len(tokenizer):,} entries | embedding size {VOCAB_SIZE:,} | marker {tokenizer.word_end_marker}')  # show tokenizer info

tokenizer: 48,000 entries | embedding size 48,000 | marker b'</w>'


### Quick encode / decode test

In [59]:
ids = tokenizer.encode("The Supreme Court dismissed the appeal filed by the petitioner.")  # turn a test sentence into token ids
print(ids); print(tokenizer.decode(ids))  # print the ids, then turn them back into text (should match the sentence)

[350, 2591, 402, 1688, 265, 745, 836, 319, 265, 7419]
The Supreme Court dismissed the appeal filed by the petitioner.


## 4. Load the CJPE data
`train.json` / `dev.json` (id, text, label) and `test.json` (id, text). Every case is tokenised with your Phase-1 tokenizer and closed with `<eos>`.

### Helpers: file lookup, streaming JSON reader, chunk offsets

In [60]:
try:  # try to import ijson
    import ijson                            # streams the (large) JSON files record by record
except ImportError:  # not installed
    import subprocess  # lets us run command-line programs
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ijson'], check=False)  # install ijson with pip
    try:  # try importing again
        import ijson  # import after installing
    except ImportError:  # still failed
        ijson = None  # we will read the files the slow way
        print('ijson not available -> the JSON files are read in one go')  # warn the user

word_cache = {}                             # the Phase-1 tokenizer works word by word, so caching per word is exact


def encode_words(words):  # turn a list of words into token ids (using the cache for speed)
    out = []  # the list of ids we will return
    for w in words:  # one word at a time
        ids = word_cache.get(w)  # have we already tokenised this word?
        if ids is None:  # no
            ids = word_cache[w] = tokenizer.encode(w)  # tokenise it and remember the result
        out.extend(ids)  # add the word's ids to the output
    return out  # all ids together


def find_data_file(name, sub):  # find a data file in one of three places
    """Local override -> attached Kaggle dataset -> Hugging Face dataset repo."""
    if LOCAL_DATA_DIR:  # if the user set a local folder
        for p in (os.path.join(LOCAL_DATA_DIR, sub, name), os.path.join(LOCAL_DATA_DIR, name)):  # try two possible paths
            if os.path.exists(p):  # does the file exist there?
                return p  # yes -> use it
    hits = [h for h in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if PHASE4_DIR.lower() in h.lower() or 'cjpe' in h.lower()]  # search Kaggle inputs for files with this name that look like the CJPE data
    if hits:  # found one
        return hits[0]  # use it
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f'{PHASE4_DIR}/{sub}/{name}', repo_type='dataset', token=HF_TOKEN)  # last option: download from Hugging Face


def iter_records(path):  # read the cases from a JSON file one by one
    """Yields the case records of a JSON list (streamed with ijson) or of a {case_id: record} dictionary."""
    with open(path, 'rb') as f:  # open the file
        first = f.read(1)  # read the first character
        while first and first.isspace():  # skip any blank spaces at the start
            first = f.read(1)  # read the next character
    if first == b'[' and ijson is not None:  # the file is a JSON list and ijson is available
        with open(path, 'rb') as f:  # open it again
            yield from ijson.items(f, 'item')  # stream the items one at a time (saves memory)
    else:  # otherwise load the whole file
        raw = json.load(open(path, encoding='utf-8'))  # read everything into memory
        if isinstance(raw, dict):  # the file is {case_id: record}
            for k, v in raw.items():  # go through each entry
                v = dict(v); v.setdefault('id', k)  # copy the record and add the key as its "id" if missing
                yield v  # give back this record
        else:  # the file is a plain list
            yield from raw  # give back each record


def chunk_starts(n, chunk_len, stride):  # where each chunk of a document begins
    """Start offsets of the overlapping chunks of a document of n tokens; the last chunk is aligned with the end of the document."""
    if n <= chunk_len:  # the document fits into one chunk
        return [0]  # just one chunk, starting at 0
    starts = list(range(0, n - chunk_len + 1, stride))  # start positions, moving forward by "stride" each time
    if starts[-1] + chunk_len < n:  # the last chunk does not reach the end of the document
        starts.append(n - chunk_len)  # add one more chunk that ends exactly at the document end
    return starts  # list of start positions


def head_tail(seq, limit, head_frac):  # shorten a sequence by keeping its start and end
    if len(seq) <= limit:  # already short enough
        return seq  # keep everything
    h = int(limit * head_frac)  # how many items to keep from the start
    return seq[:h] + seq[len(seq) - (limit - h):]  # keep the first h items and the last (limit - h) items


def load_split(path, labeled, max_docs=None):  # read a whole data file and tokenise every case
    """-> list of dict(id, tok (int32 array incl. <eos>), starts, n_words, y).  Very long cases keep their head and their tail."""
    docs = []  # list that will hold all processed documents
    for i, rec in enumerate(tqdm(iter_records(path), desc=os.path.basename(path), leave=False)):  # loop over cases with a progress bar
        if max_docs and i >= max_docs:  # stop early if we only want a few documents (smoke test)
            break  # leave the loop
        words = str(rec.get('text', '')).split()  # split the case text into words
        ids = encode_words(head_tail(words, MAX_DOC_TOKENS, HEAD_FRAC))      # a word is >= 1 token, so this never loses a needed token
        ids = head_tail(ids, MAX_DOC_TOKENS - 1, HEAD_FRAC) + [EOS]  # cut to the token limit (leaving room) and add the end-of-text token
        tok = np.asarray(ids, np.int32)  # store ids as a compact numpy array
        docs.append(dict(id=str(rec['id']), tok=tok, starts=chunk_starts(len(tok), CHUNK_LEN, CHUNK_STRIDE), n_words=len(words),
                         y=int(rec['label']) if labeled else -1))  # save id, tokens, chunk start positions, word count and label (-1 if no label)
    return docs  # list of documents

### Load train / dev / test

In [61]:
train_docs = load_split(find_data_file('train.json', 'Train'), True, MAX_DOCS)  # load and tokenise the training cases (with labels)
dev_docs   = load_split(find_data_file('dev.json', 'Train'), True, MAX_DOCS)  # load the development (validation) cases (with labels)
test_docs  = load_split(find_data_file('test.json', 'Test'), False, MAX_DOCS)  # load the test cases (no labels)
assert len({d['id'] for d in test_docs}) == len(test_docs), 'duplicate case ids in test.json'  # make sure every test id is unique
print(f'tokenizer cache: {len(word_cache):,} distinct words')  # show how many different words were tokenised

train.json: 0it [00:00, ?it/s]

dev.json: 0it [00:00, ?it/s]

test.json: 0it [00:00, ?it/s]

tokenizer cache: 787,862 distinct words


### Statistics

In [62]:
def describe(name, docs, labeled=True):  # print simple statistics of a dataset
    n_tok = np.array([len(d['tok']) for d in docs]); n_ch = np.array([len(d['starts']) for d in docs])  # tokens per case and chunks per case
    lab = f' | label counts {dict(sorted(Counter(d["y"] for d in docs).items()))}' if labeled else ''  # text showing how many cases have each label
    print(f'{name:5s}: {len(docs):6d} cases{lab}\n       tokens/case mean {n_tok.mean():7.0f} | median {np.median(n_tok):7.0f} | '
          f'90th pct {np.percentile(n_tok, 90):7.0f} | max {n_tok.max():6d} | chunks/case mean {n_ch.mean():5.1f} (max {n_ch.max()})')  # print count, average / median / 90th percentile / max length and chunk counts


describe('train', train_docs); describe('dev', dev_docs); describe('test', test_docs, labeled=False)  # show stats for the three splits

train:  37387 cases | label counts {0: 22067, 1: 15320}
       tokens/case mean    3269 | median    2443 | 90th pct    7333 | max  12288 | chunks/case mean   4.0 (max 14)
dev  :   1988 cases | label counts {0: 994, 1: 994}
       tokens/case mean    4086 | median    3301 | 90th pct    7964 | max  12288 | chunks/case mean   4.9 (max 14)
test :   1517 cases
       tokens/case mean    4195 | median    3387 | 90th pct    8304 | max  12288 | chunks/case mean   5.0 (max 14)


### Class balance and class weights
The loss is class-weighted (`CW_POWER`); the decision threshold is tuned for macro-F1 later (section 11).

In [63]:
NUM_CLASSES = 2  # two possible outcomes: label 0 or label 1
counts = np.bincount([d['y'] for d in train_docs], minlength=NUM_CLASSES).astype(np.float64)  # count training cases of each label
cw = np.ones(NUM_CLASSES, np.float32)  # start with equal weights for both classes
if CW_POWER > 0:  # if class weighting is switched on
    cw = ((counts.sum() / (NUM_CLASSES * np.maximum(counts, 1))) ** CW_POWER).astype(np.float32)  # rarer class gets a larger weight
    cw = cw / cw[counts > 0].mean()                                                      # mean weight ~1
CLASS_W = torch.tensor(cw, device=device)  # turn the weights into a tensor on the GPU/CPU
print('training cases per label / loss weight (class balancing):')  # heading for the table
for c in range(NUM_CLASSES):  # one line per label
    print(f'   label {c}: {int(counts[c]):6d} ({100 * counts[c] / counts.sum():5.1f}%)   weight {cw[c]:.3f}')  # show the count, percentage and weight

training cases per label / loss weight (class balancing):
   label 0:  22067 ( 59.0%)   weight 0.909
   label 1:  15320 ( 41.0%)   weight 1.091


## 5. Chunks and batches
A judgment becomes a list of overlapping chunks. During training a random subset of `TRAIN_CHUNKS` chunks (document order kept, last chunk always included) is read; at validation / test time up to `MAX_EVAL_CHUNKS` evenly spread chunks are read.

### Chunk selection and batch builder

In [64]:
def select_chunks(n, mode, cap, rng=None, keep_last=True):  # decide which chunks of a document the model will read
    """Indices (into the document's list of n chunks) that the model reads.
    mode: 'full' every chunk | 'first' | 'last' | 'head_tail' (first + last).  More than `cap` chunks are thinned out:
    random subset in document order while training (the last chunk is kept if keep_last), evenly spread when rng is None."""
    if mode == 'first':  # read only the first chunk
        idx = [0]
    elif mode == 'last':  # read only the last chunk
        idx = [n - 1]
    elif mode == 'head_tail':  # read the first and the last chunk
        idx = sorted({0, n - 1})  # a set avoids a duplicate when n == 1
    elif mode == 'full':  # read all chunks
        idx = list(range(n))
    else:  # anything else is a mistake
        raise ValueError(f'unknown SECTION_MODE {mode!r}')
    if len(idx) > cap:  # too many chunks -> pick only "cap" of them
        if rng is not None:  # training: pick randomly
            if keep_last:  # always keep the last chunk
                pick = sorted(rng.choice(np.array(idx[:-1]), cap - 1, replace=False).tolist()) + [idx[-1]]  # random others + the last one
            else:  # no special treatment of the last chunk
                pick = sorted(rng.choice(np.array(idx), cap, replace=False).tolist())  # purely random pick
            idx = pick  # use the picked chunks
        else:  # validation / test: pick evenly spaced chunks
            idx = sorted({idx[int(round(i))] for i in np.linspace(0, len(idx) - 1, cap)})  # evenly spread positions, first and last included
    return idx  # chunk numbers to read


def make_batch(docs, doc_ids, train, rng=None, mode=None, max_chunks=None):  # build the tensors for a group of documents
    """Documents -> one tensor of chunks.  ids/mask: (N_chunks, T) | doc_idx/slot: which document / which slot of it each chunk is |
    pos_bin: relative position of the chunk in its judgment | y: label of every document"""
    mode = mode or SECTION_MODE  # default to the global SECTION_MODE
    max_chunks = max_chunks or (TRAIN_CHUNKS if train else MAX_EVAL_CHUNKS)  # default number of chunks depends on training or evaluation
    pieces, doc_idx, slot, pos_bin, rel = [], [], [], [], []  # lists we fill: token pieces, which doc, which slot, position bin, relative position
    for b, di in enumerate(doc_ids):  # b = position in the batch, di = document number
        d = docs[di]  # get the document
        starts = d['starts']  # chunk start positions of this document
        sel = select_chunks(len(starts), mode, max_chunks, rng if train else None, TRAIN_KEEP_LAST)  # which chunks to read
        rel.append(np.array([starts[i] / max(1, starts[-1]) for i in sel], np.float32))  # relative position (0 = start, 1 = end) of each chosen chunk
        for j, i in enumerate(sel):  # go through each chosen chunk
            pieces.append(d['tok'][starts[i]:starts[i] + CHUNK_LEN])  # cut out the tokens of this chunk
            doc_idx.append(b); slot.append(j)  # remember which document and which slot this chunk belongs to
            pos_bin.append(int(round(rel[-1][j] * (N_POS_BINS - 1))))  # turn the relative position into a bin number
    T = max(len(p) for p in pieces)  # the longest chunk in this batch
    ids = np.full((len(pieces), T), PAD_ID, np.int64); mask = np.zeros((len(pieces), T), bool)  # token matrix filled with padding, and a mask (True = real token)
    for r, p in enumerate(pieces):  # copy each chunk into the matrix
        ids[r, :len(p)], mask[r, :len(p)] = p, True  # real tokens first, then padding
    to = lambda a: torch.from_numpy(a).to(device)  # helper: numpy array -> tensor on the device
    return dict(ids=to(ids), mask=to(mask), doc_idx=to(np.array(doc_idx, np.int64)), slot=to(np.array(slot, np.int64)),
                pos_bin=to(np.array(pos_bin, np.int64)), n_docs=len(doc_ids), max_slots=int(max(slot)) + 1, rel=rel,
                y=to(np.array([docs[i]['y'] for i in doc_ids], np.int64)))  # pack everything into one dictionary (the "batch")

## 6. Metrics and error analysis
* **Macro-F1** (primary), macro precision and recall, per-label scores and the confusion matrix;
* decision-threshold tuning for macro-F1 and a calibration check (ECE);
* characteristics of **false positives / false negatives**: size, document length, confidence, most confident mistakes.

### Macro-F1 / precision / recall, threshold tuning, calibration

In [65]:
def binary_scores(y_true, y_pred):  # compute all the scores for a 2-class problem
    """Macro-F1 (primary), macro precision / recall, accuracy, confusion matrix cm[gold, predicted] and per-label scores."""
    cm = np.bincount(y_true * 2 + y_pred, minlength=4).reshape(2, 2)  # confusion matrix: rows = true label, columns = predicted label
    tp, sup, prd = np.diag(cm).astype(float), cm.sum(1), cm.sum(0)  # correct predictions, true count per label, predicted count per label
    p, r = tp / np.maximum(prd, 1), tp / np.maximum(sup, 1)  # precision and recall per label (max(.,1) avoids dividing by zero)
    f = 2 * p * r / np.maximum(p + r, 1e-12)  # F1 score per label
    return dict(macro_f1=float(f.mean()), macro_p=float(p.mean()), macro_r=float(r.mean()), accuracy=float(tp.sum() / max(1, cm.sum())),
                cm=cm.tolist(), per_label={c: dict(p=float(p[c]), r=float(r[c]), f1=float(f[c]), support=int(sup[c])) for c in range(2)})  # macro = simple average over the two labels


def tune_threshold(y_true, p1, grid=np.linspace(0.05, 0.95, 91)):  # find the best cut-off probability
    """Decision threshold on P(label = 1) that maximises macro-F1 (post-processing for the class imbalance)."""
    best_t, best_f = 0.5, -1.0  # best threshold and score so far
    for t in grid:  # try many thresholds from 0.05 to 0.95
        f = binary_scores(y_true, (p1 >= t).astype(int))['macro_f1']  # score if we predict 1 when probability >= t
        if f > best_f + 1e-12:  # better than before?
            best_t, best_f = float(t), f  # remember it
    return best_t, best_f  # the best threshold and its macro-F1


def expected_calibration_error(y_true, probs, bins=10):  # measures if confidence matches accuracy
    conf, pred = probs.max(1), probs.argmax(1)  # the model's confidence and its predicted label
    ece, edges = 0.0, np.linspace(0, 1, bins + 1)  # running error and the edges of the confidence bins
    for lo, hi in zip(edges[:-1], edges[1:]):  # go through each confidence bin
        m = (conf > lo) & (conf <= hi)  # which predictions fall in this bin
        if m.any():  # bin is not empty
            ece += m.mean() * abs((pred[m] == y_true[m]).mean() - conf[m].mean())  # add (bin size) x |accuracy - confidence|
    return float(ece)  # lower is better


def print_scores(s, title=''):  # nicely print the score dictionary
    print(f'--- {title} macro-F1 {s["macro_f1"]:.4f} | macro-P {s["macro_p"]:.4f} | macro-R {s["macro_r"]:.4f} | accuracy {s["accuracy"]:.4f}')  # overall scores
    for c in (0, 1):  # per-label scores
        l = s['per_label'][c]  # scores of label c
        print(f'      label {c}: precision {l["p"]:.4f} | recall {l["r"]:.4f} | F1 {l["f1"]:.4f} | support {l["support"]}')  # print them
    cm = np.array(s['cm'])  # confusion matrix as an array
    print(f'      confusion (rows = gold, columns = predicted):  [[{cm[0, 0]:5d} {cm[0, 1]:5d}]  [{cm[1, 0]:5d} {cm[1, 1]:5d}]]')  # print the 2x2 matrix

### False-positive / false-negative analysis

In [66]:
def error_analysis(docs, probs, thr, top=5):  # study what kind of mistakes the model makes
    """Characteristics of the four outcome groups (label 1 = positive class): size, document length, confidence, and the most confident mistakes."""
    y = np.array([d['y'] for d in docs]); p1 = probs[:, 1]; pred = (p1 >= thr).astype(int)  # true labels, probability of label 1, and predicted labels
    n_tok = np.array([len(d['tok']) for d in docs]); n_ch = np.array([len(d['starts']) for d in docs])  # tokens and chunks per document
    groups = {'TP': (y == 1) & (pred == 1), 'TN': (y == 0) & (pred == 0), 'FP': (y == 0) & (pred == 1), 'FN': (y == 1) & (pred == 0)}  # true/false positives and negatives
    out = dict(groups={}, by_length={}, most_confident_errors={})  # dictionary to store results
    print(f'{"group":6s}{"cases":>7s}{"tokens/case":>13s}{"chunks/case":>13s}{"mean P(1)":>11s}')  # table header
    for name, m in groups.items():  # one row per group
        out['groups'][name] = dict(count=int(m.sum()), mean_tokens=float(n_tok[m].mean()) if m.any() else None,
                                   mean_chunks=float(n_ch[m].mean()) if m.any() else None, mean_p1=float(p1[m].mean()) if m.any() else None)  # count and averages (None if the group is empty)
        g = out['groups'][name]  # the stats just stored
        print(f'{name:6s}{g["count"]:7d}' + ''.join(f'{g[k]:{w}.{q}f}' if g[k] is not None else f'{"-":>{w}s}'
                                                       for k, w, q in (('mean_tokens', 13, 0), ('mean_chunks', 13, 1), ('mean_p1', 11, 3))))  # print the row ("-" when empty)
    edges = np.quantile(n_tok, [0, .25, .5, .75, 1.0])  # boundaries that split documents into 4 equal-size length groups
    print('error behaviour by document length (quartiles of tokens):')  # heading
    for q in range(4):  # go through each length quartile
        m = (n_tok >= edges[q]) & ((n_tok <= edges[q + 1]) if q == 3 else (n_tok < edges[q + 1]))  # documents inside this length range
        if m.sum() == 0:  # no documents in this range
            continue  # skip it
        s = binary_scores(y[m], pred[m])  # scores just for these documents
        out['by_length'][f'Q{q + 1}'] = dict(tokens=f'{int(edges[q])}-{int(edges[q + 1])}', cases=int(m.sum()), macro_f1=s['macro_f1'], accuracy=s['accuracy'])  # store them
        print(f'   Q{q + 1} ({int(edges[q]):6d}-{int(edges[q + 1]):6d} tokens, {int(m.sum()):4d} cases): accuracy {s["accuracy"]:.3f} | macro-F1 {s["macro_f1"]:.3f}')  # print them
    for name in ('FP', 'FN'):  # look at both kinds of mistakes
        idx = np.where(groups[name])[0]  # positions of these mistakes
        conf = (p1 if name == 'FP' else 1 - p1)[idx]  # how confident the model was in the wrong answer
        pick = idx[np.argsort(-conf)[:top]]  # the "top" most confident mistakes
        out['most_confident_errors'][name] = [dict(id=docs[i]['id'], p1=float(p1[i]), tokens=int(n_tok[i])) for i in pick]  # store their id, probability and length
        print(f'most confident {name}:', [(docs[i]['id'], round(float(p1[i]), 3)) for i in pick])  # print them
    return out  # return everything for saving

### Diagnostics: duplicates / leakage, and a TF-IDF reference model
If the TF-IDF model scores well above chance, the labels are recoverable from the wording; its probabilities are also blended in section 12g.

In [67]:
# ---- duplicates and train/dev/test overlap ----
tr_ids = {d['id'] for d in train_docs}; dv_ids = {d['id'] for d in dev_docs}; te_ids = {d['id'] for d in test_docs}
print('duplicate ids   train / dev / test:', len(train_docs) - len(tr_ids), '/', len(dev_docs) - len(dv_ids), '/', len(test_docs) - len(te_ids))
print('overlap         dev in train:', len(dv_ids & tr_ids), '| test in train:', len(te_ids & tr_ids), '| test in dev:', len(te_ids & dv_ids))
for d in train_docs[:3]:                                                    # does the END of a judgment reveal the outcome?
    print('label', d['y'], '| last tokens:', tokenizer.decode(d['tok'][-120:].tolist()), '\n')

duplicate ids   train / dev / test: 5082 / 994 / 0
overlap         dev in train: 0 | test in train: 0 | test in dev: 0
label 0 | last tokens: of the minority educational institutions on any companynt. In the premises, while allowing these appeals, we set aside the view taken by the Single Judge and the Division Bench of the High Court and dismiss Writ Petition No.20650 W of 2013 and other companynected matters. We also hold Sections 8, 10, 11 and 12 of the Commission Act to be valid and companystitutional. In the end, we declare all numberinations made by the Commission in pursuance of the provisions of the Commission Act to be valid and Civil Appeal No.5808 of 2017 SK. MD. Rafique vs. Managing Committee, companytai Rahamania High Madrasah and Others operative. 

label 0 | last tokens: hypothesis than that of the guilt of the accused and such evidence should number only be companysistent with the guilt of the accused but should be inconsistent with his innocence. Keeping the above test

In [68]:
# ---- TF-IDF + logistic regression reference (CPU, a few minutes) ----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def tail(t, n=2500, head=500):                                              # keep the start and the (informative) end of a long case
    w = str(t).split()
    return ' '.join(w[:head] + w[-n:]) if len(w) > n + head else ' '.join(w)

def read_texts(name, sub, labeled):
    X, y = [], []
    for i, r in enumerate(iter_records(find_data_file(name, sub))):
        if MAX_DOCS and i >= MAX_DOCS: break
        X.append(tail(r.get('text', ''))); y.append(int(r['label']) if labeled else -1)
    return X, y

tf_tr, ytf_tr = read_texts('train.json', 'Train', True)
tf_dv, ytf_dv = read_texts('dev.json', 'Train', True)
tf_te, _ = read_texts('test.json', 'Test', False)
tfidf_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=300000, sublinear_tf=True, dtype=np.float32)
Xtr = tfidf_vec.fit_transform(tf_tr); Xdv = tfidf_vec.transform(tf_dv); Xte = tfidf_vec.transform(tf_te)
tfidf_clf = LogisticRegression(C=4, max_iter=300, class_weight='balanced').fit(Xtr, ytf_tr)
tfidf_dev, tfidf_test = tfidf_clf.predict_proba(Xdv), tfidf_clf.predict_proba(Xte)
assert len(tfidf_dev) == len(dev_docs) and len(tfidf_test) == len(test_docs)
ydv = np.array([d['y'] for d in dev_docs])
_thr, _f = tune_threshold(ydv, tfidf_dev[:, 1]) if 'tune_threshold' in globals() else (0.5, float('nan'))
print(f'TF-IDF dev macro-F1 @0.5: {f1_score(ydv, tfidf_dev.argmax(1), average="macro"):.4f} | best-threshold: {_f:.4f} (thr {_thr:.2f})')
del tf_tr, Xtr; gc.collect()

TF-IDF dev macro-F1 @0.5: 0.6837 | best-threshold: 0.6952 (thr 0.44)


85

## 7. Your Phase-1 model
The architecture code is copied **unchanged** from your Phase-1 notebook; the weights come from your checkpoint.

### Model code (hand-written, unchanged from Phase 1)

In [69]:
import math  # maths functions (sqrt, cos, ...)
from dataclasses import dataclass  # for the Config class below
from typing import List, Optional  # type hints
import torch.nn as nn  # neural network layers
import torch.nn.functional as F  # functional operations
from torch.utils.checkpoint import checkpoint  # gradient checkpointing: saves memory by recomputing values later

### Config

In [70]:
@dataclass  # automatically creates __init__ etc. from the fields below
class Config:  # all the settings that describe the Phase-1 model
    vocab_size: int = 32000  # number of different tokens
    d_model: int = 768  # size of the vector that represents each token
    n_layer: int = 12  # number of transformer blocks
    n_head: int = 12  # number of attention heads
    n_kv_head: Optional[int] = None      # None -> MHA; < n_head -> GQA
    max_seq_len: int = 2048  # longest sequence the model can handle
    rope_theta: float = 10000.0  # base number used by rotary position encoding
    rope_pct: float = 1.0                # fraction of the head dim that is rotated (partial RoPE)
    dropout: float = 0.0  # chance of randomly switching off values while training (0 = off)
    tie_embeddings: bool = False  # share weights between input embedding and output layer
    qk_norm: bool = False  # normalise queries and keys inside attention
    grad_ckpt: bool = False  # use gradient checkpointing to save memory
    value_residual: bool = False         # later layers mix in the first layer's values
    mtp_heads: int = 0                   # extra multi-token-prediction heads (t+2, t+3, ...)
    mtp_weight: float = 0.3  # importance of the extra prediction heads in the loss
    n_sources: int = 0                   # >0 -> source embedding (SC/Acts/HC/Tribunal) + 1 "unknown"
    src_drop: float = 0.3                # prob. of replacing the source id by "unknown" in training

    @property
    def head_dim(self):  # size of one attention head
        return self.d_model // self.n_head  # split d_model equally among heads

    @property
    def kv_heads(self):  # number of key/value heads
        return self.n_kv_head or self.n_head  # if not set, use the same as the number of heads

    @property
    def ff_dim(self):                    # SwiGLU hidden size, parameter-matched to a 4*d GELU MLP
        return int(round(8 / 3 * self.d_model / 64)) * 64  # about 8/3 of d_model, rounded to a multiple of 64

### Building blocks: RMSNorm, RoPE, SwiGLU MLP

In [71]:
class RMSNorm(nn.Module):  # a normalisation layer: keeps numbers at a stable size
    def __init__(self, d, eps=1e-6):  # d = vector size, eps = tiny number to avoid dividing by zero
        super().__init__()  # set up the parent nn.Module
        self.w = nn.Parameter(torch.ones(d))  # learnable scale for each dimension (starts at 1)
        self.eps = eps  # remember eps

    def forward(self, x):  # runs the layer
        xf = x.float()  # use 32-bit numbers for accuracy
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)  # divide by the root-mean-square of the vector
        return (out * self.w.float()).type_as(x)  # apply learnable scale and go back to the original number type


class Rotary(nn.Module):  # RoPE: encodes token positions by rotating vectors
    def __init__(self, dim, max_len, theta):  # dim = size being rotated, max_len = longest sequence, theta = base number
        super().__init__()  # set up the parent class
        inv = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / max(dim, 1)))  # a different rotation speed for each pair of dimensions
        freqs = torch.outer(torch.arange(max_len).float(), inv)  # angle for every (position, speed) combination
        self.register_buffer("cos", freqs.cos(), persistent=False)  # store cosines (not learned, not saved)
        self.register_buffer("sin", freqs.sin(), persistent=False)  # store sines (not learned, not saved)

    def forward(self, T, start=0):  # get cos/sin for T positions
        """cos/sin for positions start .. start+T-1 (start > 0 only while generating with a cache)."""
        return self.cos[start:start + T], self.sin[start:start + T]  # cut out the needed positions


def apply_rope(x, cos, sin, rot_dim):  # rotate the vectors x using cos and sin
    """x: (B,H,T,hd). Rotates the first rot_dim dims (half-split convention)."""
    if rot_dim == 0:  # nothing to rotate
        return x  # return unchanged
    xr, xp = x[..., :rot_dim], x[..., rot_dim:]  # xr = part that is rotated, xp = part left as it is
    x1, x2 = xr.float().chunk(2, dim=-1)  # split the rotated part into two halves
    c, s = cos[None, None], sin[None, None]  # add batch and head dimensions so shapes match
    xr = torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)  # the 2D rotation formula
    return torch.cat([xr, xp], dim=-1)  # put rotated and untouched parts back together


class MLP(nn.Module):  # the feed-forward part of a transformer block
    """SwiGLU feed-forward."""
    def __init__(self, cfg):  # cfg = model settings
        super().__init__()  # set up the parent class
        d, f = cfg.d_model, cfg.ff_dim  # input size and hidden size
        self.gate = nn.Linear(d, f, bias=False)  # layer that makes the "gate" values
        self.up = nn.Linear(d, f, bias=False)  # layer that makes the "content" values
        self.down = nn.Linear(f, d, bias=False)  # layer that shrinks back to size d

    def forward(self, x):  # runs the layer
        return self.down(F.silu(self.gate(x)) * self.up(x))  # gate (through SiLU) x content, then shrink back

### KV cache (new: lets generation process one token per step instead of the whole prefix)

In [72]:
# --------------------------------------------------------------------------------------
# KV cache + attention
# --------------------------------------------------------------------------------------
class KVCache:  # remembers past keys/values so generation is faster
    """Pre-allocated key/value buffer for ONE attention layer (stored un-expanded, i.e. Hkv heads).
    Lets generation process one new token per step instead of re-running the whole prefix."""

    def __init__(self, max_len):  # max_len = biggest total length we will store
        self.max_len, self.n, self.k, self.v = max_len, 0, None, None  # n = how many tokens stored so far; k, v are created on first use

    def append(self, k, v):  # add new keys and values
        if self.k is None:  # first time: create empty buffers
            B, H, _, hd = k.shape  # batch size, heads, (length), head size
            self.k = k.new_empty(B, H, self.max_len, hd)  # empty space for keys
            self.v = v.new_empty(B, H, self.max_len, hd)  # empty space for values
        T = k.size(2)  # number of new tokens
        self.k[:, :, self.n:self.n + T] = k  # write new keys after the old ones
        self.v[:, :, self.n:self.n + T] = v  # write new values after the old ones
        self.n += T  # update how many tokens are stored

    def get(self, window=None):  # read back the stored keys/values
        s = 0 if window is None else max(0, self.n - window)  # start index (only the last "window" tokens if a window is set)
        return self.k[:, :, s:self.n], self.v[:, :, s:self.n]  # return the stored part


def sdpa_gqa(q, k, v, mask=None, causal=False, p=0.0):  # the attention calculation (supports grouped-query attention)
    """q: (B,H,T,hd)   k,v: (B,Hkv,S,hd).  Handles GQA without copying the cache when T == 1."""
    B, H, T, hd = q.shape  # batch, query heads, length, head size
    Hkv = k.size(1)  # number of key/value heads
    if T == 1 and mask is None:                       # single-token decode step
        if Hkv != H:                                  # fold the query-head group into the "length" axis
            r = H // Hkv  # how many query heads share one key/value head
            out = F.scaled_dot_product_attention(q.reshape(B, Hkv, r, hd), k, v)  # attention with grouped queries
            return out.reshape(B, H, 1, hd)  # restore the original shape
        return F.scaled_dot_product_attention(q, k, v)  # normal attention
    if Hkv != H:  # fewer key/value heads than query heads
        r = H // Hkv  # group size
        k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)  # copy keys/values so every query head has a partner
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p,
                                          is_causal=causal and mask is None)  # attention (causal = tokens only look at earlier tokens)

### Self-attention (GQA / QK-norm / RoPE-or-NoPE / optional value residual, with KV-cache support)

In [73]:
class Attention(nn.Module):  # self-attention: lets every token look at other tokens
    """MHA/GQA self-attention with optional RoPE (or NoPE), QK-norm, value residual, sliding-window mask
    and KV cache.  Returns (output, v_of_the_new_tokens)."""

    def __init__(self, cfg, causal=True, use_rope=True, value_residual=False):  # build the layers
        super().__init__()  # set up the parent class
        d = cfg.d_model  # token vector size
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim  # number of heads, key/value heads, head size
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"  # heads must split evenly into groups
        self.q = nn.Linear(d, self.H * self.hd, bias=False)  # makes queries
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)  # makes keys
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)  # makes values
        self.o = nn.Linear(self.H * self.hd, d, bias=False)  # output layer that mixes heads together
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None  # optional normalisation of queries
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None  # optional normalisation of keys
        self.p_drop = cfg.dropout  # dropout chance inside attention
        self.causal = causal  # True = tokens cannot see the future
        self.rot_dim = (int(self.hd * cfg.rope_pct) // 2 * 2) if use_rope else 0  # how many dimensions get rotated (even number)
        self.lam = nn.Parameter(torch.zeros(self.Hkv, 1, 1)) if value_residual else None  # learnable mixing weight for the value residual

    def forward(self, x, rope=None, v_first=None, mask=None, cache=None, window=None):  # runs attention
        B, T, _ = x.shape  # batch size and sequence length
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)  # queries, shaped (B, heads, T, head_size)
        k = self.k(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)  # keys
        v = self.v(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)  # values
        if self.lam is not None and v_first is not None:  # value residual is active
            lam = torch.sigmoid(self.lam).to(v.dtype)  # mixing weight between 0 and 1
            v = (1 - lam) * v + lam * v_first  # blend in the first layer's values
        v_new = v  # keep these values to return
        if self.q_norm is not None:  # QK-norm is on
            q, k = self.q_norm(q), self.k_norm(k)  # normalise queries and keys
        if self.rot_dim:  # rotary positions are on
            cos, sin = rope  # unpack the cos/sin tables
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)  # rotate queries and keys by position
        causal = self.causal  # copy the causal setting
        if cache is not None:  # generation mode with a KV cache
            cache.append(k, v)  # store the new keys and values
            k, v = cache.get(window if T == 1 else None)   # window only matters for 1-token steps
            causal = T > 1                                  # prefill: causal; decode step: sees everything cached
        out = sdpa_gqa(q, k, v, mask=mask, causal=causal, p=self.p_drop if self.training else 0.0)  # do the attention calculation
        return self.o(out.transpose(1, 2).reshape(B, T, -1)), v_new  # merge heads, apply the output layer, return result and values

### LM head (final norm + tied unembedding + multi-token-prediction heads)

In [74]:
class LMHead(nn.Module):  # the part that turns hidden states into word predictions
    """Final norm + unembedding (optionally tied) + optional multi-token-prediction heads."""

    def __init__(self, cfg, tok_emb: nn.Embedding):  # cfg = settings, tok_emb = the token embedding table
        super().__init__()  # set up the parent class
        self.cfg = cfg  # keep the settings
        self.norm = RMSNorm(cfg.d_model)  # final normalisation
        self.proj = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)  # layer that gives a score for every token in the vocabulary
        if cfg.tie_embeddings:  # share weights with the input embedding?
            self.proj.weight = tok_emb.weight  # yes: use the same weight matrix
        self.mtp = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False)
                                  for _ in range(getattr(cfg, "mtp_heads", 0))])  # layers for predicting tokens further in the future

    def logits(self, h):  # hidden states -> token scores
        return self.proj(self.norm(h))  # normalise, then project to the vocabulary

    def forward(self, h, targets=None):  # compute scores and (if targets given) the loss
        h = self.norm(h)  # normalise the hidden states
        logits = self.proj(h)  # scores for every token
        if targets is None:  # no targets -> no loss needed
            return logits, None
        V = logits.size(-1)  # vocabulary size
        loss = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1), ignore_index=-100)  # next-token prediction loss
        self.main_loss = loss.detach()          # next-token loss only (without MTP terms), for logging
        w = getattr(self.cfg, "mtp_weight", 0.3)  # weight of extra-prediction losses
        for k, p in enumerate(self.mtp, start=1):  # each extra head predicts k tokens ahead
            if targets.size(1) <= k:  # sequence too short for this head
                break  # stop
            hk = h[:, :-k] + F.silu(p(h[:, :-k]))  # transform hidden states for the k-ahead prediction
            lk = self.proj(hk)  # scores for the k-ahead prediction
            loss = loss + w * F.cross_entropy(lk.reshape(-1, V).float(), targets[:, k:].reshape(-1),
                                              ignore_index=-100)  # add this head's loss, scaled by w
        return logits, loss  # return scores and total loss

### Helpers: gradient checkpointing, weight init, base class

In [75]:
def ckpt_call(fn, enabled, *args):  # run a function, optionally with gradient checkpointing
    return checkpoint(fn, *args, use_reentrant=False) if enabled else fn(*args)  # checkpoint = save memory by recomputing later


def init_weights(model, depth):  # set starting values for the model's weights
    for m in model.modules():  # go through every sub-module
        if isinstance(m, (nn.Linear, nn.Embedding)):  # only linear and embedding layers
            nn.init.normal_(m.weight, 0.0, 0.02)  # small random numbers (mean 0, std 0.02)
    std = 0.02 / math.sqrt(2 * depth)  # smaller std for output layers of deep models
    for n, p in model.named_parameters():  # go through every parameter
        if n.endswith(("o.weight", "down.weight")):  # the layers that write back into the residual stream
            nn.init.normal_(p, 0.0, std)  # use the smaller std for them


class BaseLM(nn.Module):  # parent class with shared helpers for language models
    """Common interface used by training.py / generation.py:
         forward(x, targets=None, src=None) -> (logits, loss)
         prefill(prompt_ids, max_new)       -> (logits_for_next_token, state)      [generation]
         step(token_ids, state)             -> logits_for_next_token               [generation]
       `eval_offset` = number of leading label positions that are NOT scored (encoder-decoder only)."""
    eval_offset = 0  # no label positions are skipped for a decoder

    def num_params(self, non_embedding=False):  # count the model's parameters
        n = sum(p.numel() for p in self.parameters())  # total number of values
        return n - self.tok.weight.numel() if non_embedding else n  # optionally leave out the embedding table

### Transformer block

In [76]:
class Block(nn.Module):  # one transformer block = attention + feed-forward
    """Pre-norm residual block (attention + SwiGLU)."""

    def __init__(self, cfg, value_residual=False):  # build the parts
        super().__init__()  # set up the parent class
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)  # two normalisation layers (before attention, before MLP)
        self.attn = Attention(cfg, value_residual=value_residual)  # attention part
        self.mlp = MLP(cfg)  # feed-forward part
        self.drop = nn.Dropout(cfg.dropout)  # dropout layer

    def forward(self, x, rope, v_first=None, cache=None):  # runs the block
        h, v = self.attn(self.n1(x), rope, v_first, cache=cache)  # normalise, then attention
        x = x + self.drop(h)  # add attention output back to x (residual connection)
        return x + self.drop(self.mlp(self.n2(x))), v  # normalise, feed-forward, add back, and return the values too

### Decoder-only model

In [77]:
class DecoderLM(BaseLM):  # the full Phase-1 decoder language model
    def __init__(self, cfg: Config):  # build all parts
        super().__init__()  # set up the parent class
        self.cfg = cfg  # keep the settings
        self.tok = nn.Embedding(cfg.vocab_size, cfg.d_model)  # table that turns token ids into vectors
        self.src_emb = nn.Embedding(cfg.n_sources + 1, cfg.d_model) if cfg.n_sources else None  # optional "source" embedding (+1 for unknown)
        self.rope = Rotary(int(cfg.head_dim * cfg.rope_pct) // 2 * 2, cfg.max_seq_len, cfg.rope_theta)  # rotary position tables
        self.blocks = nn.ModuleList([Block(cfg, cfg.value_residual and i > 0) for i in range(cfg.n_layer)])  # stack of transformer blocks
        self.head = LMHead(cfg, self.tok)  # output head
        init_weights(self, cfg.n_layer)  # set starting weights

    def hidden(self, idx, src=None, caches=None, start=0):  # token ids -> hidden states
        cfg = self.cfg  # shortcut to settings
        B, T = idx.shape  # batch size and length
        x = self.tok(idx)  # look up token vectors
        if self.src_emb is not None:  # model uses a source embedding
            unk = cfg.n_sources  # id meaning "unknown source"
            if src is None:                                  # unknown source (validation / test)
                src = torch.full((B,), unk, device=idx.device, dtype=torch.long)  # mark all as unknown
            elif self.training and cfg.src_drop > 0:  # while training, sometimes hide the source
                drop = torch.rand(B, device=idx.device) < cfg.src_drop  # randomly choose which to hide
                src = torch.where(drop, torch.full_like(src, unk), src)  # replace chosen ones with "unknown"
            x = x + self.src_emb(src)[:, None, :]  # add the source vector to every token
        rope = self.rope(T, start)  # get position tables for this length
        v_first = None  # will hold the first layer's values
        for i, blk in enumerate(self.blocks):  # pass through each block in turn
            if caches is None:  # normal (training / classification) mode
                x, v = ckpt_call(blk, cfg.grad_ckpt and self.training, x, rope, v_first)  # run the block (with checkpointing if enabled)
            else:  # generation mode
                x, v = blk(x, rope, v_first, cache=caches[i])  # run the block using its cache
            if v_first is None and cfg.value_residual:  # first layer just ran
                v_first = v  # remember its values for later layers
        return x  # final hidden states

    def forward(self, idx, targets=None, src=None):  # full forward pass
        return self.head(self.hidden(idx, src), targets)  # hidden states -> logits (and loss)

    # ---- generation (KV cache) ----
    def prefill(self, idx, max_new):  # process the prompt for generation
        caches = [KVCache(idx.shape[1] + max_new) for _ in self.blocks]  # one cache per block
        h = self.hidden(idx, caches=caches)  # run the prompt and fill the caches
        return self.head.logits(h[:, -1]), caches  # scores for the next token, plus the caches

    def step(self, tok, caches):  # generate one more token
        h = self.hidden(tok, caches=caches, start=caches[0].n)  # run only the new token, at the right position
        return self.head.logits(h[:, -1])  # scores for the next token

### Baseline / modified architecture definitions

In [78]:
Model = DecoderLM  # a short alias for the model class


BASE = dict(d_model=768, n_layer=12, n_head=12, max_seq_len=2048)  # common settings of all architectures


def cfg_baseline(**kw):  # settings of the plain baseline model
    return Config(**{**BASE, **kw})  # BASE settings, overridden by any extra settings


def cfg_modified(**kw):  # settings of the modified model (the one used here)
    return Config(**{**BASE, "n_layer": 17, "n_kv_head": 4, "qk_norm": True, "tie_embeddings": True,
                     "value_residual": True, "mtp_heads": 2, "n_sources": 4, **kw})  # 17 layers + GQA + QK-norm + tied embeddings + value residual + 2 MTP heads + 4 sources

### Locate and load the Phase-1 checkpoint

In [79]:
def find_ckpt():  # find the Phase-1 checkpoint file
    if PHASE1_CKPT and os.path.exists(PHASE1_CKPT):  # the user gave a valid path
        return PHASE1_CKPT  # use it
    cands = sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True))  # find every .pt file in Kaggle inputs
    norm = lambda s: s.lower().replace('-', '').replace('_', '')  # helper: lower-case and drop "-" and "_" so names compare easily
    hit = [c for c in cands if norm(CKPT_HINT) in norm(c)] or cands  # files whose path contains the hint (or all if none match)
    if not hit:  # nothing found
        raise FileNotFoundError('No Phase-1 checkpoint found: attach it as a Kaggle dataset or set PHASE1_CKPT')  # stop with a clear message
    print('checkpoint candidates:', *hit, sep='\n   ')  # list the candidates
    return ([c for c in hit if 'best' in os.path.basename(c)] or [c for c in hit if 'last' in os.path.basename(c)] or hit)[0]  # prefer "best", then "last", else the first


ckpt_path = find_ckpt()  # get the checkpoint path
PHASE1 = torch.load(ckpt_path, map_location='cpu', weights_only=False)     # {'model', 'cfg', 'step', 'val_loss'}
D_MODEL = PHASE1['cfg']['d_model']  # the model's vector size
ARCH_NAME = 'arch1_decoder'  # a name for this architecture
assert VOCAB_SIZE <= PHASE1['cfg']['vocab_size'], 'the tokenizer is larger than the checkpoint vocabulary: wrong tokenizer / checkpoint pair'  # tokenizer ids must fit in the model's embedding table
print('using', ckpt_path, '| Phase-1 step', PHASE1.get('step'), '| val loss', PHASE1.get('val_loss'))  # show what was loaded

checkpoint candidates:
   /kaggle/input/datasets/sunnysfhbv/model-arch-1/last.pt
using /kaggle/input/datasets/sunnysfhbv/model-arch-1/last.pt | Phase-1 step 3688 | val loss 2.346502783942584


### Rebuild the Phase-1 model and read out its hidden states
How the model is used here: causal decoder. The states of three intermediate layers (`TAP_LAYERS`) are read; per layer, the **mean** of the token states and the state of the **last** token of a chunk form the chunk vector.

In [80]:
def load_phase1_backbone(keep_layers=None):  # rebuild the Phase-1 model and load its trained weights
    """Rebuilds the Phase-1 model from the checkpoint (model classes above are unchanged) and drops the LM output layer.
    keep_layers: optional list of layer indices -> a shallower model made of those Phase-1 layers (used for the distillation student)."""
    valid = {f.name for f in fields(Config)}  # names of all valid Config settings
    cfg = Config(**{k: v for k, v in PHASE1['cfg'].items() if k in valid})  # build a Config from the checkpoint (ignore unknown keys)
    cfg.grad_ckpt, cfg.dropout = GRAD_CKPT, BACKBONE_DROPOUT  # apply our own memory-saving and dropout settings
    m = DecoderLM(cfg)  # create the empty model
    missing, unexpected = m.load_state_dict(PHASE1['model'], strict=False)  # copy the trained weights into it
    assert not missing, f'missing keys when loading Phase-1 weights: {missing[:5]}'  # make sure no weights are missing
    m.head.proj, m.head.mtp = nn.Identity(), nn.ModuleList()                # LM output layer / MTP heads are not needed
    if keep_layers is not None:  # build a shallower model?
        m.blocks = nn.ModuleList([m.blocks[i] for i in keep_layers])  # keep only the chosen blocks
        m.cfg.n_layer = len(keep_layers)  # update the layer count
    return m  # return the model


def get_block_lists(m):  # find where the transformer blocks are
    """The ModuleLists of transformer blocks that the classifier uses (LoRA / LoRMA / adapters are attached to these)."""
    return [m.blocks]  # this model has just one list of blocks


def unused_modules(m):  # parts that the classifier never uses
    """Parts of the Phase-1 model that the classifier never calls (they get no gradient)."""
    return []  # none for this architecture


def prompt_len(m):  # how many soft-prompt vectors are attached
    return getattr(m.tok, 'n_prompt', 0)                                    # > 0 when a soft prompt is attached


SOURCE_ID = 0       # Phase-1 source ids: 0 = Supreme Court, 1 = Acts, 2 = High Courts, 3 = Tribunals  (CJPE cases are Supreme Court judgments); None = "unknown"


TAP_LAYERS = (8, 12, 16)      # 0-indexed blocks whose states are pooled (17-layer Phase-1 model)


def backbone_features(m, ids, mask):  # token ids -> concatenated hidden states of several layers
    """(N, T) token ids -> (N, T, len(TAP_LAYERS) * d) scale-free hidden states of intermediate layers.
    The last layer of a language model is specialised for next-token prediction; middle layers carry more general document information."""
    assert max(TAP_LAYERS) < len(m.blocks), f'TAP_LAYERS {TAP_LAYERS} need at least {max(TAP_LAYERS) + 1} blocks (model has {len(m.blocks)})'
    p = prompt_len(m)  # number of soft-prompt positions (0 if none)
    if p:  # soft prompt is used
        ids = F.pad(ids, (p, 0), value=PAD_ID)                              # placeholders that PromptedEmbedding replaces by the soft prompt
    cfg = m.cfg  # model settings
    x = m.tok(ids)  # token vectors
    if m.src_emb is not None and SOURCE_ID is not None:  # the model has a source embedding
        src = torch.full((ids.size(0),), SOURCE_ID, device=ids.device, dtype=torch.long)  # every case is a Supreme Court case
        x = x + m.src_emb(src)[:, None, :]  # add the source vector to every token
    rope = m.rope(ids.size(1))  # rotary position tables
    v_first, outs = None, []  # first layer's values (value residual) and collected layer outputs
    for i, blk in enumerate(m.blocks):  # run the blocks one by one
        x, v = ckpt_call(blk, cfg.grad_ckpt and m.training, x, rope, v_first)  # same call as DecoderLM.hidden
        if v_first is None and cfg.value_residual:  # first block just ran
            v_first = v
        if i in TAP_LAYERS:  # keep this layer's states
            xf = x.float()
            outs.append(xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + 1e-6))   # RMS-normalise (no learnable weight)
    return torch.cat(outs, -1)[:, p:]  # drop the soft-prompt positions


FEAT_DIM = 2 * D_MODEL * len(TAP_LAYERS)                                    # chunk vector = [mean | last token] x tapped layers


### Check: parameters of the Phase-1 model

In [81]:
_probe = load_phase1_backbone()  # load the model just to inspect it
print(f'{ARCH_NAME}: {sum(p.numel() for p in _probe.parameters()) / 1e6:.1f}M parameters loaded from Phase 1 (d_model {D_MODEL})')  # print number of parameters in millions
del _probe  # free the memory

arch1_decoder: 143.9M parameters loaded from Phase 1 (d_model 768)


## 8. Adaptation methods (choose with `METHOD` in the configuration)
The techniques of Assignment 4, re-written for the Phase-1 `nn.Linear` layers.

| `METHOD` | what is trained | idea (Assignment 4) |
|---|---|---|
| `'lorma'` | low-rank matrices `A`, `B` (**compulsory method**) | LoRMA+: `W' = (I + (alpha/r)·B·A)·W0`, computed as `z = W0·x`, `y = z + (alpha/r)·B·(A·z)` (no `d x d` matrix is built) |
| `'lora'` | low-rank matrices `A`, `B` | additive: `W' = W0 + (alpha/r)·B·A` |
| `'adapter'` | a bottleneck MLP after every block | `h' = h + W_up·GELU(W_down·h)`, `W_up = 0` at the start |
| `'soft_prompt'` | learned input vectors | `N_PROMPT` trainable embeddings replace the first positions of every chunk |
| `'full'` | every Phase-1 parameter | full fine-tuning (reference, and the distillation student) |

`B` (LoRA / LoRMA) and `W_up` (adapter) start at zero, so every method starts exactly at the Phase-1 model. The Phase-1 weights are frozen first; the document head is always trained.

### Soft prompt, adapter, LoRA, LoRMA

In [82]:
class SoftPrompt(nn.Module):  # a small set of learnable vectors placed in front of the input
    """Assignment 4.1: trainable continuous vectors that replace the first `m` input positions (initialised from real token embeddings)."""
    def __init__(self, init_embeddings):                       # [m, d_model]
        super().__init__()  # set up the parent class
        self.prompt = nn.Parameter(init_embeddings.detach().clone().float())  # copy the starting vectors and make them learnable


class PromptedEmbedding(nn.Module):  # embedding layer that inserts the soft prompt
    """Wraps the frozen token embedding.  The caller puts `m` placeholder ids in front of every sequence; their embeddings are replaced by the
    learned prompt, so every position-dependent part of the (unchanged) Phase-1 model sees a sequence that is simply m tokens longer."""
    def __init__(self, emb, soft):  # emb = original embedding, soft = the soft prompt
        super().__init__()  # set up the parent class
        self.emb, self.soft = emb, soft  # keep both
        self.n_prompt = soft.prompt.shape[0]  # how many prompt vectors there are

    @property
    def weight(self):  # lets other code read the embedding weights as usual
        return self.emb.weight  # the original embedding table

    def forward(self, ids):  # ids -> vectors
        x = self.emb(ids)  # normal embeddings for all positions
        prompt = self.soft.prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1)  # repeat the prompt for every item in the batch
        return torch.cat([prompt, x[:, self.n_prompt:]], dim=1)  # replace the first positions by the prompt


class BottleneckAdapter(nn.Module):  # a tiny extra network added after a block
    """Assignment 4.2:  h' = h + W_up GELU(W_down h + b_down) + b_up,  with W_up and b_up zero-initialised (identity at step 0)."""
    def __init__(self, hidden_size, bottleneck=16):  # hidden_size = vector size, bottleneck = small middle size
        super().__init__()  # set up the parent class
        self.down, self.up = nn.Linear(hidden_size, bottleneck), nn.Linear(bottleneck, hidden_size)  # shrink, then expand
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)  # start the "up" layer at zero so the adapter initially changes nothing

    def forward(self, h):  # runs the adapter
        return h + self.up(F.gelu(self.down(h)))  # add a small learned correction to h


class AdaptedBlock(nn.Module):  # a block with an adapter attached after it
    """A Phase-1 block followed by an adapter (a wrapper module instead of a forward hook, so that nn.DataParallel replicas use their own adapter)."""
    def __init__(self, block, adapter):  # the original block and its adapter
        super().__init__()  # set up the parent class
        self.block, self.adapter = block, adapter  # keep both

    def forward(self, *args, **kwargs):  # runs block then adapter
        out = self.block(*args, **kwargs)  # run the original block
        if isinstance(out, tuple):                              # some blocks return (hidden, extra)
            return (self.adapter(out[0]),) + tuple(out[1:])  # adapt the hidden state, keep the extras
        return self.adapter(out)  # adapt the output

    def __getattr__(self, name):                                # e.g. `block.window` of the sliding-window architecture
        try:  # first look in this wrapper itself
            return super().__getattr__(name)
        except AttributeError:  # not found here
            if name == 'block' or name.startswith('__'):  # avoid endless loops for special names
                raise  # pass the error on
            return getattr(self.block, name)  # look inside the wrapped block instead


class LoRALinear(nn.Module):  # a linear layer with a LoRA update
    """Assignment 4.3 (LoRA):  y = W0 x + s * B (A x),  A [r, d_in] random, B [d_out, r] zero,  s = alpha / r."""
    def __init__(self, base, r, alpha, dropout=0.0):  # base = original layer, r = rank, alpha = scale factor
        super().__init__()  # set up the parent class
        self.base, self.scale = base, alpha / r  # keep the original layer and compute the scale
        self.A = nn.Parameter(torch.empty(r, base.in_features))  # small matrix A (random start)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))  # small matrix B (zero start, so no change at first)
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # fill A with random numbers
        self.drop = nn.Dropout(dropout)  # dropout on the input of the LoRA path

    def forward(self, x):  # runs the layer
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)  # original output + small low-rank correction


def make_lorma_factors(d_out, rank, generator=None, dtype=torch.float32):  # create the A and B matrices for LoRMA
    """Assignment 4.4: base-preserving initialisation, A [r, d_out] ~ U(-1/sqrt(d_out), 1/sqrt(d_out)), B [d_out, r] = 0."""
    bound = 1.0 / math.sqrt(d_out)  # range for the random numbers
    return ((torch.rand(rank, d_out, generator=generator, dtype=torch.float64) * 2 - 1) * bound).to(dtype), torch.zeros(d_out, rank, dtype=dtype)  # A = random in [-bound, bound], B = zeros


def lorma_dense(x, W0, factors, scale):  # slow, literal version of the LoRMA formula (used only for testing)
    """Literal LoRMA_+ equation  W' = (s B A + I) W0  (a d_out x d_out matrix is materialised) -> x W'^T.  W0: [d_out, d_in]."""
    A, B = factors  # unpack the two small matrices
    return x @ ((scale * (B @ A) + torch.eye(W0.shape[0], dtype=B.dtype, device=B.device)) @ W0).T  # build the full matrix, then apply it


def lorma_reordered(x, W0, factors, scale):  # fast version of the same formula
    """The same function without any d x d product:  h = x W0^T ;  y = h + s (h A^T) B^T."""
    A, B = factors  # unpack the two small matrices
    h = x @ W0.T  # first apply the original frozen weights
    return h + scale * ((h @ A.T) @ B.T)  # then add the small low-rank correction


class LoRMALinear(nn.Module):  # a linear layer with a LoRMA (multiplicative) update
    """LoRMA_+ (multiplicative low-rank adaptation, Assignment 4.4) around a frozen nn.Linear.  Only A and B are trained."""
    def __init__(self, base, r, alpha):  # base = original layer, r = rank, alpha = scale factor
        super().__init__()  # set up the parent class
        self.base, self.scale = base, alpha / r  # keep the original layer and compute the scale
        A, B = make_lorma_factors(base.out_features, r)  # create the starting A and B
        self.A, self.B = nn.Parameter(A), nn.Parameter(B)  # make them learnable

    def forward(self, x):  # runs the layer
        y = lorma_reordered(x, self.base.weight, (self.A, self.B), self.scale)  # LoRMA output (without bias)
        return y if self.base.bias is None else y + self.base.bias  # add the bias if the layer has one


def wrap_linears(module, targets, make):  # replace chosen linear layers with adapted versions
    """Replaces every nn.Linear called `targets` (q, k, v, o, ...) inside `module` by make(linear); returns how many were replaced."""
    n = 0  # counter of replaced layers
    for name, child in list(module.named_children()):  # go through every direct child module
        if isinstance(child, nn.Linear) and name in targets:  # a linear layer with a target name
            setattr(module, name, make(child)); n += 1  # swap it for the adapted version
        else:  # not a target
            n += wrap_linears(child, targets, make)  # look deeper inside it
    return n  # how many were replaced


def prompt_init_embeddings(backbone, n_prompt):  # starting vectors for the soft prompt
    ids = tokenizer.encode(PROMPT_INIT_TEXT) or [PAD_ID]  # tokenise the starting text (use padding id if empty)
    ids = [ids[i % len(ids)] for i in range(n_prompt)]  # repeat/cut the ids to exactly n_prompt items
    return backbone.tok.weight.detach().float().cpu()[torch.tensor(ids)]  # look up the embeddings of those ids


def apply_method(backbone, hp):  # set up the chosen adaptation method on the model
    """Freezes the Phase-1 weights, then makes a small part trainable according to hp['method'] (the document head is always trained)."""
    method = hp['method']  # which method to use
    for p in backbone.parameters():  # go through all Phase-1 weights
        p.requires_grad_(False)  # freeze them (they will not be updated)
    block_lists = get_block_lists(backbone)  # find the transformer blocks
    if method in ('lora', 'lorma'):  # low-rank methods
        make = (lambda l: LoRALinear(l, hp['r'], hp['alpha'], hp['dropout'])) if method == 'lora' else (lambda l: LoRMALinear(l, hp['r'], hp['alpha']))  # choose which wrapper to use
        n = sum(wrap_linears(blk, set(hp['targets']), make) for ml in block_lists for blk in ml)  # wrap the target layers in every block
        assert n > 0, f'no linear layer called {hp["targets"]} found'  # make sure something was wrapped
    elif method == 'adapter':  # adapter method
        for ml in block_lists:  # for each list of blocks
            for i in range(len(ml)):  # for each block
                ml[i] = AdaptedBlock(ml[i], BottleneckAdapter(D_MODEL, hp['bottleneck']))  # attach an adapter after it
    elif method == 'soft_prompt':  # soft prompt method
        backbone.tok = PromptedEmbedding(backbone.tok, SoftPrompt(prompt_init_embeddings(backbone, hp['n_prompt'])))  # wrap the embedding with a soft prompt
    elif method == 'full':  # full fine-tuning
        for p in backbone.parameters():  # all weights
            p.requires_grad_(True)  # unfreeze everything
    else:  # unknown method name
        raise ValueError(f'unknown METHOD {method!r}')
    for mod in unused_modules(backbone):                         # parts of the Phase-1 model that the classifier never calls
        for p in mod.parameters():  # their weights
            p.requires_grad_(False)  # keep them frozen

### Sanity checks

In [83]:
# sanity checks of the adaptation code (a few seconds on CPU)
torch.manual_seed(0)  # fixed seed so the test is repeatable
_lin = nn.Linear(24, 20, bias=False).double()  # a small test layer (64-bit numbers for precision)
_x = torch.randn(3, 5, 24, dtype=torch.double)  # random test input
_f = make_lorma_factors(20, 4, dtype=torch.double)  # fresh LoRMA matrices (B is zero)
assert torch.allclose(lorma_reordered(_x, _lin.weight, _f, 2.0), _x @ _lin.weight.T), 'LoRMA must start exactly at the Phase-1 layer'  # with B = 0 the output must equal the original layer
_f = (torch.randn(4, 20, dtype=torch.double) * 0.3, torch.randn(20, 4, dtype=torch.double) * 0.3)  # now use random non-zero A and B
assert torch.allclose(lorma_reordered(_x, _lin.weight, _f, 2.0), lorma_dense(_x, _lin.weight, _f, 2.0)), 'reordered LoRMA != literal equation'  # fast and slow versions must agree
_l = nn.Linear(24, 20, bias=True)  # a test layer with a bias
assert torch.allclose(LoRALinear(_l, 4, 8)(_x.float()), _l(_x.float())) and torch.allclose(LoRMALinear(_l, 4, 8)(_x.float()), _l(_x.float()), atol=1e-6)  # LoRA and LoRMA must start equal to the original layer
_h = torch.randn(2, 7, 24); assert torch.equal(BottleneckAdapter(24, 4)(_h), _h)  # a new adapter must not change its input
print('adaptation checks passed: LoRMA / LoRA / adapter all start exactly at the Phase-1 model; LoRMA reordered == dense equation')  # all tests passed

adaptation checks passed: LoRMA / LoRA / adapter all start exactly at the Phase-1 model; LoRMA reordered == dense equation


## 9. Document classifier
Phase-1 model (adapted) -> chunk vectors -> hierarchical document head -> outcome.

### Chunk encoder, document head, model builder

In [84]:
class ChunkEncoder(nn.Module):  # turns each chunk of tokens into one vector
    """Phase-1 model -> one vector per chunk: [mean of the token states | state of the last real token]."""
    def __init__(self, backbone):  # backbone = the (adapted) Phase-1 model
        super().__init__()  # set up the parent class
        self.backbone = backbone  # keep it

    def forward(self, ids, mask):  # ids = tokens, mask = which are real tokens
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):  # use faster low-precision maths on the GPU
            h = backbone_features(self.backbone, ids, mask)                           # (N, T, d_token)
        h = h.float()  # go back to 32-bit numbers
        m = mask.unsqueeze(-1).float()  # mask with an extra dimension (1 = real token, 0 = padding)
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1)  # average of the real tokens' vectors
        last = h[torch.arange(h.size(0), device=h.device), (mask.sum(1) - 1).clamp(min=0)]  # vector of the last real token
        return torch.cat([mean, last], -1)                                            # (N, 2 * d_token)


class DocHead(nn.Module):  # the small network that turns chunk vectors into a final answer
    """Hierarchical document head: chunk vectors (+ embedding of the chunk's relative position = section-aware) -> pooling -> classifier.
    pool: 'attn' learned attention pooling | 'mean' | 'gru' (BiGRU over the chunk sequence, then mean) | 'last' (last chunk only)."""
    def __init__(self, feat_dim, hidden, n_cls, pool, n_bins):  # sizes and the pooling type
        super().__init__()  # set up the parent class
        self.pool = pool  # remember the pooling type
        self.proj = nn.Sequential(nn.LayerNorm(feat_dim), nn.Linear(feat_dim, hidden), nn.GELU())  # normalise and shrink chunk vectors
        self.pos = nn.Embedding(n_bins, hidden)  # embedding for the chunk's position in the document
        if pool == 'attn':  # attention pooling
            self.att = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1))  # network that scores how important each chunk is
        elif pool == 'gru':  # GRU pooling
            self.gru = nn.GRU(hidden, hidden // 2, batch_first=True, bidirectional=True)  # reads chunks forwards and backwards
        elif pool not in ('mean', 'last'):  # anything else is invalid
            raise ValueError(f'unknown DOC_POOL {pool!r}')
        self.out = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(0.1), nn.Linear(hidden, n_cls))  # final small classifier
        self.last_w = None                                                            # attention weights of the last forward pass (analysis)

    def forward(self, feats, batch):  # feats = chunk vectors
        z = self.proj(feats) + self.pos(batch['pos_bin'])                             # (N, h)
        B, K = batch['n_docs'], batch['max_slots']  # number of documents and max chunks per document
        pad = z.new_zeros(B, K, z.size(-1)); pad[batch['doc_idx'], batch['slot']] = z  # put the chunk vectors into a (document, slot) grid
        valid = torch.zeros(B, K, dtype=torch.bool, device=z.device); valid[batch['doc_idx'], batch['slot']] = True  # mark which grid cells hold real chunks
        n = valid.sum(1)  # number of real chunks per document
        if self.pool == 'mean':  # average pooling
            v = (pad * valid.unsqueeze(-1)).sum(1) / n.unsqueeze(-1)  # mean of the real chunk vectors
        elif self.pool == 'attn':  # attention pooling
            w = torch.softmax(self.att(pad).squeeze(-1).masked_fill(~valid, float('-inf')), 1)  # importance weights that add up to 1 (empty cells ignored)
            self.last_w = w.detach()  # save weights for analysis
            v = (w.unsqueeze(-1) * pad).sum(1)  # weighted sum of chunk vectors
        elif self.pool == 'gru':  # GRU pooling
            packed = nn.utils.rnn.pack_padded_sequence(pad, n.cpu(), batch_first=True, enforce_sorted=False)  # pack sequences so padding is skipped
            g, _ = nn.utils.rnn.pad_packed_sequence(self.gru(packed)[0], batch_first=True, total_length=K)  # run the GRU and unpack
            v = (g * valid.unsqueeze(-1)).sum(1) / n.unsqueeze(-1)  # mean of the GRU outputs
        else:  # 'last'
            v = pad[torch.arange(B, device=z.device), n - 1]  # take the last real chunk
        return self.out(v)  # document vector -> class scores


class CJPEModel(nn.Module):  # the complete classifier
    """Phase-1 model (adapted) -> chunk vectors -> document head -> logits for the 2 outcomes.  forward(batch) -> (logits, chunk_vectors)."""
    def __init__(self, backbone, feat_dim, hp):  # backbone, chunk-vector size, settings
        super().__init__()  # set up the parent class
        self.enc = ChunkEncoder(backbone)  # part 1: chunk encoder
        self.head = DocHead(feat_dim, hp['hidden'], NUM_CLASSES, hp['pool'], N_POS_BINS)  # part 2: document head
        self.hp = hp  # keep the settings

    def use_data_parallel(self):  # use several GPUs if available
        """Splits the chunks of a batch over all GPUs (the document head stays on GPU 0).  Not registered as a sub-module on purpose."""
        if N_GPU > 1:  # more than one GPU
            object.__setattr__(self, '_dp', nn.DataParallel(self.enc))  # wrap the encoder so work is shared between GPUs

    def forward(self, batch):  # runs the whole classifier
        enc = getattr(self, '_dp', None) or self.enc  # use the multi-GPU encoder if it exists
        feats = enc(batch['ids'], batch['mask'])  # tokens -> chunk vectors
        return self.head(feats, batch), feats  # chunk vectors -> class scores (also return the vectors)


def make_hp(method=None, **kw):  # collect all model settings in one dictionary
    """All settings that define a model, in one dictionary (also saved with the weights)."""
    hp = dict(method=method or METHOD, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT, targets=list(PEFT_TARGETS), bottleneck=ADAPTER_BOTTLENECK,
              n_prompt=N_PROMPT, pool=DOC_POOL, hidden=HEAD_HIDDEN, keep_layers=None)  # default values from the configuration section
    hp.update(kw)  # override with any extra values
    return hp  # the settings dictionary


def count_params(model):  # count the model's parameters
    total = sum(p.numel() for p in model.parameters())  # all parameters
    unused = sum(p.numel() for mod in unused_modules(model.enc.backbone) for p in mod.parameters())  # parameters never used
    return dict(total=total, forward_path=total - unused, trainable=sum(p.numel() for p in model.parameters() if p.requires_grad))  # total, used, and trainable counts


def build_model(hp, tag=''):  # create the whole model ready for training
    """Fresh Phase-1 backbone -> adaptation method -> document head."""
    backbone = load_phase1_backbone(hp['keep_layers'])  # load the Phase-1 model
    apply_method(backbone, hp)  # freeze it and add the adaptation (e.g. LoRMA)
    model = CJPEModel(backbone, FEAT_DIM, hp).to(device)  # add the document head and move to the GPU
    model.use_data_parallel()  # use multiple GPUs if there are any
    c = count_params(model)  # count the parameters
    print(f'{tag}{hp["method"]} | pool={hp["pool"]} | {c["total"] / 1e6:.1f}M parameters ({c["forward_path"] / 1e6:.1f}M used by the classifier), '
          f'{c["trainable"] / 1e6:.2f}M trainable ({100 * c["trainable"] / c["total"]:.2f}%)')  # print the sizes
    return model, c  # return the model and the counts

## 10. Training code
Loss = class-weighted cross-entropy with label smoothing. AdamW, warm-up then cosine decay (stretched to the time cap), gradient clipping, fp16 / bf16 autocast, quick validation every `EVAL_EVERY` steps, best weights kept.

### Optimizer, trainable-weight snapshots, prediction, quick validation

In [85]:
def make_optimizer(model, lr, head_lr):  # create the optimizer (the thing that updates weights)
    head = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('head.')]  # trainable parameters of the document head
    body = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('head.')]  # trainable parameters of the backbone (LoRMA etc.)
    groups = [dict(params=head, lr=head_lr, base_lr=head_lr)] + ([dict(params=body, lr=lr, base_lr=lr)] if body else [])  # two groups, each with its own learning rate
    return torch.optim.AdamW(groups, weight_decay=0.01)  # AdamW optimizer with a little weight decay


def lr_factor(progress, warmup=0.10):  # learning-rate schedule: warm-up then slow decay
    if progress < warmup:  # at the very start
        return (progress + 1e-3) / warmup  # increase the rate gradually from ~0 to 1
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, (progress - warmup) / (1 - warmup))))  # then decrease along a cosine curve down to 5%


def get_trainable_state(model):  # take a copy of the trainable weights
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}  # name -> copied values


def load_trainable_state(model, state):  # put saved weights back into the model
    params = dict(model.named_parameters())  # name -> parameter
    with torch.no_grad():  # do not track gradients while copying
        for n, v in state.items():  # for every saved weight
            params[n].copy_(v)  # copy it in


@torch.no_grad()  # no gradients needed when predicting (saves memory)
def predict_docs(model, docs, mode=None, max_chunks=None, want_attn=False):  # get probabilities for many documents
    """-> probs (n_docs, 2), info(tokens read, attention weights per document).  Documents of similar size are batched together."""
    mode, max_chunks = mode or SECTION_MODE, max_chunks or MAX_EVAL_CHUNKS  # use defaults if not given
    was_training = model.training  # remember if the model was in training mode
    model.eval()  # switch to evaluation mode (no dropout)
    sizes = [len(select_chunks(len(d['starts']), mode, max_chunks)) for d in docs]  # number of chunks each document will use
    order = sorted(range(len(docs)), key=lambda i: sizes[i])  # sort documents by size so batches are similar
    probs, n_tok, attn, i = np.zeros((len(docs), NUM_CLASSES), np.float32), 0, {}, 0  # result array, token counter, attention storage, position pointer
    while i < len(order):  # until all documents are done
        ids, c = [], 0  # documents in this batch and their chunk count
        while i < len(order) and (not ids or c + sizes[order[i]] <= EVAL_CHUNKS_PER_BATCH):  # add documents while the batch is not too big
            ids.append(order[i]); c += sizes[order[i]]; i += 1  # add one document
        batch = make_batch(docs, ids, False, mode=mode, max_chunks=max_chunks)  # build the tensors
        logits, _ = model(batch)  # run the model
        probs[ids] = F.softmax(logits.float(), -1).cpu().numpy()  # turn scores into probabilities and store them
        n_tok += int(batch['mask'].sum())  # count tokens read
        if want_attn and model.head.pool == 'attn':  # attention weights were requested
            w = model.head.last_w.cpu().numpy()  # get the weights
            for b, di in enumerate(ids):  # for each document in the batch
                attn[di] = (batch['rel'][b], w[b, :len(batch['rel'][b])])  # save positions and weights
    model.train(was_training)  # restore the previous mode
    return probs, dict(tokens=n_tok, attn=attn)  # probabilities and extra info


quick_rng = np.random.RandomState(SEED)  # random generator for choosing the quick-validation documents
quick_dev = [dev_docs[i] for i in sorted(quick_rng.choice(len(dev_docs), min(QUICK_DEV_DOCS, len(dev_docs)), replace=False))]  # a fixed random subset of the dev set


def quick_eval(model):  # fast validation used during training
    """Selection score during training: macro-F1 on a dev subset (at the best threshold if APPLY_TUNED_THRESHOLD, else at 0.5)."""
    probs, _ = predict_docs(model, quick_dev, max_chunks=QUICK_EVAL_CHUNKS)  # predict the small dev subset
    y = np.array([d['y'] for d in quick_dev])  # true labels
    return tune_threshold(y, probs[:, 1])[1] if APPLY_TUNED_THRESHOLD else binary_scores(y, probs.argmax(1))['macro_f1']  # macro-F1 with tuned or default threshold

### Training loop

In [86]:
def cjpe_loss(logits, y):  # the training loss
    return F.cross_entropy(logits.float(), y, weight=CLASS_W, label_smoothing=LABEL_SMOOTH)  # cross-entropy with class weights and label smoothing


def run_training(model, tag, epochs, lr, head_lr, teacher=None, kd_loss_fn=None):  # the main training function
    """Trains `model` (and keeps the dev-best weights).  teacher / kd_loss_fn: optional knowledge distillation (used by the KD notebook)."""
    opt = make_optimizer(model, lr, head_lr)  # create the optimizer
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda' and amp_dtype == torch.float16))  # helps fp16 training stay stable
    params = [p for g in opt.param_groups for p in g['params']]  # list of all trained parameters
    pool_size = len(train_docs) if not EPOCH_DOCS else min(EPOCH_DOCS, len(train_docs))  # how many documents per epoch
    docs_per_step = BATCH_DOCS * GRAD_ACCUM  # documents seen per optimizer step
    steps_per_epoch = max(1, pool_size // docs_per_step)  # optimizer steps in one epoch
    total_steps = min(steps_per_epoch * epochs, MAX_STEPS or 10 ** 9)  # total steps (limited by MAX_STEPS if set)
    best, history, bad, step, run_loss, run_n = dict(score=-1.0, step=0, state=None), [], 0, 0, 0.0, 0  # best result so far, log, bad-validation counter, step counter, running loss
    t0, done = time.time(), False  # start time and "finished" flag
    if teacher is not None:  # distillation only
        teacher.eval()  # the teacher stays in evaluation mode
    if device == 'cuda':  # on GPU
        torch.cuda.reset_peak_memory_stats()  # start measuring peak memory from zero
    print(f'[{tag}] {steps_per_epoch} optimizer steps per epoch ({docs_per_step} documents each), {total_steps} in total, time cap {MAX_TRAIN_MINUTES} min')  # print the plan
    model.train()  # switch to training mode
    pbar = tqdm(total=total_steps, desc=tag, dynamic_ncols=True)  # progress bar
    for epoch in range(epochs):  # loop over epochs
        rng = np.random.RandomState(SEED + epoch)  # a different random order each epoch
        perm = rng.permutation(len(train_docs))[:pool_size]  # shuffled list of document numbers
        for s in range(steps_per_epoch):  # loop over optimizer steps
            progress = max(step / total_steps, (time.time() - t0) / (MAX_TRAIN_MINUTES * 60))  # progress by steps or by time, whichever is further
            if progress >= 1.0:  # finished
                done = True; break  # stop training
            for g in opt.param_groups:  # for each parameter group
                g['lr'] = g['base_lr'] * lr_factor(progress)  # set the learning rate for this moment
            for m in range(GRAD_ACCUM):  # accumulate gradients over several small batches
                idx = perm[(s * GRAD_ACCUM + m) * BATCH_DOCS:(s * GRAD_ACCUM + m + 1) * BATCH_DOCS]  # documents for this small batch
                batch = make_batch(train_docs, idx, True, rng)  # build the tensors (training mode: random chunks)
                logits, feats = model(batch)  # run the model
                if teacher is None:  # normal training
                    loss = cjpe_loss(logits, batch['y'])  # compare with the true labels
                else:  # distillation
                    with torch.no_grad():  # the teacher is not trained
                        t_logits, t_feats = teacher(batch)  # teacher's outputs
                    loss = kd_loss_fn(logits, feats, t_logits, t_feats, batch['y'])  # loss that also copies the teacher
                scaler.scale(loss / GRAD_ACCUM).backward()  # compute gradients (divided so the sum equals the average)
                run_loss += float(loss.detach()) / GRAD_ACCUM  # add to the running loss for logging
            scaler.unscale_(opt)  # undo fp16 scaling before clipping
            nn.utils.clip_grad_norm_(params, 1.0)  # stop gradients from becoming too big
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)  # update the weights and clear the gradients
            step += 1; run_n += 1; pbar.update(1)  # count the step
            if step % LOG_EVERY == 0:  # time to log
                pbar.set_postfix(loss=f'{run_loss / run_n:.4f}', lr=f'{opt.param_groups[-1]["lr"]:.1e}')  # show the average loss and learning rate
                history.append(dict(step=step, train_loss=run_loss / run_n)); run_loss, run_n = 0.0, 0  # save it and reset the counters
            if step % EVAL_EVERY == 0 or step == total_steps:  # time to validate
                f1 = quick_eval(model); model.train()  # quick validation, then back to training mode
                history.append(dict(step=step, quick_dev_macro_f1=f1))  # save the score
                improved = f1 > best['score']  # is it the best so far?
                if improved:  # yes
                    best.update(score=f1, step=step, state=get_trainable_state(model))  # save these weights as the best
                bad = 0 if improved else bad + 1  # count validations without improvement
                pbar.write(f'[{tag}] step {step}/{total_steps} | quick dev macro-F1 {f1:.4f}' + ('  <- best' if improved else '') +
                           f' | {(time.time() - t0) / 60:.1f} min')  # print the result
                if bad >= PATIENCE and step >= 0.3 * total_steps:  # no improvement for too long
                    done = True; break  # early stopping
            if step >= total_steps:  # reached the planned number of steps
                done = True; break  # stop
        if done:  # leave the epoch loop too
            break
    pbar.close()  # close the progress bar
    if best['state'] is None:                                   # no validation happened (very short run)
        best.update(score=quick_eval(model), step=step, state=get_trainable_state(model))  # validate once now and save the weights
    load_trainable_state(model, best['state'])  # restore the best weights
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else 0.0  # peak GPU memory in GB
    print(f'[{tag}] finished in {(time.time() - t0) / 60:.1f} min | best step {best["step"]} | best quick dev macro-F1 {best["score"]:.4f} | peak GPU memory {peak:.1f} GB')  # print the summary
    return dict(history=history, best_step=best['step'], best_quick_dev_f1=best['score'], seconds=time.time() - t0, peak_mem_gb=peak, steps=step)  # return the training record

## 11. Evaluation and submission helpers

### Validation report, section analysis, attention by position, submission writer

In [87]:
def validate(model, tag, docs=None):  # full report on the development set
    """Full development report: default threshold, tuned threshold, calibration, error analysis, inference speed."""
    docs = docs if docs is not None else dev_docs  # use the whole dev set by default
    y = np.array([d['y'] for d in docs])  # true labels
    t0 = time.time()  # start timer
    probs, info = predict_docs(model, docs, want_attn=True)  # predict all documents (and save attention weights)
    tok_per_s = info['tokens'] / max(time.time() - t0, 1e-9)  # speed: tokens processed per second
    s_default = binary_scores(y, probs.argmax(1))  # scores using the normal 0.5 cut-off
    thr, f_tuned = tune_threshold(y, probs[:, 1])  # find the best cut-off
    s_tuned = binary_scores(y, (probs[:, 1] >= thr).astype(int))  # scores using the tuned cut-off
    print_scores(s_default, f'[{tag}] threshold 0.5   :')  # print default scores
    print_scores(s_tuned, f'[{tag}] threshold {thr:.2f}  :')  # print tuned scores
    ece = expected_calibration_error(y, probs)  # check how well confidence matches accuracy
    print(f'[{tag}] calibration: ECE {ece:.4f} | inference speed {tok_per_s:,.0f} tokens/s')  # print calibration and speed
    errors = error_analysis(docs, probs, thr if APPLY_TUNED_THRESHOLD else 0.5)  # analyse the mistakes
    return dict(default=s_default, tuned=s_tuned, threshold=thr, ece=ece, infer_tok_per_s=tok_per_s, errors=errors, probs=probs, attn=info['attn'])  # return everything


def section_ablation(model, tag, docs=None):  # test which part of a judgment matters
    """Which part of a judgment carries the signal?  The trained model is evaluated while it only reads the first chunk, the last chunk,
    first + last, or the whole judgment (no retraining)."""
    docs = docs if docs is not None else dev_docs  # use the dev set by default
    y = np.array([d['y'] for d in docs]); out = {}  # true labels and an empty results dictionary
    print(f'[{tag}] what the model reads        macro-F1 @0.5   best-threshold macro-F1   tokens read')  # table header
    for mode in ('first', 'last', 'head_tail', 'full'):  # try each reading mode
        probs, info = predict_docs(model, docs, mode=mode)  # predict while reading only that part
        f_default = binary_scores(y, probs.argmax(1))['macro_f1']  # macro-F1 at 0.5
        thr, f_best = tune_threshold(y, probs[:, 1])  # macro-F1 at the best threshold
        out[mode] = dict(macro_f1=f_default, macro_f1_tuned=f_best, threshold=thr, tokens_read=info['tokens'])  # store the results
        print(f'    {mode:10s}                       {f_default:.4f}               {f_best:.4f}              {info["tokens"]:,}')  # print the row
    return out  # return all results


def attention_by_position(attn, bins=5):  # which positions in a judgment get the most attention
    """Average attention weight (document head, 'attn' pooling) per relative position in the judgment, relative to a uniform weight."""
    if not attn:  # no attention data
        return None  # nothing to do
    acc, cnt = np.zeros(bins), np.zeros(bins)  # running totals and counts per position bin
    for rel, w in attn.values():  # for every document
        if len(w) < 2:  # a single chunk tells us nothing about position
            continue  # skip it
        b = np.minimum((rel * bins).astype(int), bins - 1)  # which bin each chunk falls in
        np.add.at(acc, b, w * len(w)); np.add.at(cnt, b, 1)  # add the weight (scaled so 1.0 = uniform) and count
    share = acc / np.maximum(cnt, 1)  # average per bin
    print('attention weight by position in the judgment (1.0 = uniform):', ' | '.join(f'{100 * i // bins}-{100 * (i + 1) // bins}%: {share[i]:.2f}' for i in range(bins)))  # print each bin
    return share.tolist()  # return as a list


def write_submission(test_docs, probs, thr, out_dir):  # create the files to submit
    """prediction.json = [{"id": ..., "pred": 0/1}, ...] in the order of test.json, plus submission.zip containing it."""
    pred = (probs[:, 1] >= thr).astype(int)  # predict 1 when probability reaches the threshold
    rows = [dict(id=d['id'], pred=int(p)) for d, p in zip(test_docs, pred)]  # one {id, pred} entry per test case
    assert len(rows) == len(test_docs) and all(r['pred'] in (0, 1) and isinstance(r['pred'], int) for r in rows)  # check the format is correct
    path = os.path.join(out_dir, 'prediction.json')  # where to save the JSON
    json.dump(rows, open(path, 'w', encoding='utf-8'), indent=1)  # write the predictions
    with zipfile.ZipFile(os.path.join(out_dir, 'submission.zip'), 'w', zipfile.ZIP_DEFLATED) as z:  # create a zip file
        z.write(path, arcname='prediction.json')  # put prediction.json inside it
    print(f'wrote {path} and submission.zip | {len(rows)} predictions | predicted label counts {dict(Counter(int(p) for p in pred))} | threshold {thr:.2f}')  # confirm what was saved


def summary_table(rows):  # print a comparison table of experiments
    """rows = [(name, param counts, train result, validation result)] -> compact comparison used for the report"""
    print(f'{"experiment":26s}{"params":>9s}{"fwd path":>10s}{"trainable":>11s}{"macro-F1":>10s}{"tuned F1":>10s}{"macro-P":>9s}{"macro-R":>9s}{"tok/s":>9s}{"min":>7s}{"GPU GB":>8s}')  # header row
    for name, c, tr, v in rows:  # one row per experiment
        s = v['tuned'] if APPLY_TUNED_THRESHOLD else v['default']  # which scores to show for precision/recall
        print(f'{name:26s}{c["total"] / 1e6:8.1f}M{c["forward_path"] / 1e6:9.1f}M{c["trainable"] / 1e6:10.2f}M{v["default"]["macro_f1"]:10.4f}{v["tuned"]["macro_f1"]:10.4f}'
              f'{s["macro_p"]:9.4f}{s["macro_r"]:9.4f}{v["infer_tok_per_s"]:9.0f}{tr["seconds"] / 60:7.1f}{tr["peak_mem_gb"]:8.1f}')  # print sizes, scores, speed, time and memory

## 12. Run the experiment

### 12-pre. Overfit sanity test
Trains a fresh copy on 64 documents. The training loss must fall clearly below 0.69 (chance); if it does not, the pipeline (not the hyper-parameters) is broken. Set `RUN_OVERFIT_TEST = False` to skip.

In [88]:
RUN_OVERFIT_TEST = True
if RUN_OVERFIT_TEST and not SMOKE_TEST:
    _bak = (train_docs, EVAL_EVERY, EPOCH_DOCS)
    try:
        train_docs, EVAL_EVERY, EPOCH_DOCS = train_docs[:64], 10 ** 9, None
        m_t, _ = build_model(HP, tag='[overfit] ')
        r_t = run_training(m_t, 'overfit', 20, LR, HEAD_LR)
        losses = [h['train_loss'] for h in r_t['history'] if 'train_loss' in h]
        print('training loss every', LOG_EVERY, 'steps:', [round(x, 3) for x in losses])
        print('OVERFIT TEST', 'PASSED' if losses and losses[-1] < 0.45 else 'FAILED - the loss did not go down; do not run the full training yet')
        del m_t
    finally:
        train_docs, EVAL_EVERY, EPOCH_DOCS = _bak
        gc.collect(); torch.cuda.empty_cache()

[overfit] lorma | pool=attn | 152.7M parameters (152.7M used by the classifier), 8.81M trainable (5.77%)
[overfit] 8 optimizer steps per epoch (8 documents each), 160 in total, time cap 300 min


overfit:   0%|          | 0/160 [00:00<?, ?it/s]

[overfit] step 160/160 | quick dev macro-F1 0.4960  <- best | 10.5 min
[overfit] finished in 10.5 min | best step 160 | best quick dev macro-F1 0.4960 | peak GPU memory 3.0 GB
training loss every 20 steps: [0.617, 0.234, 0.144, 0.131, 0.128, 0.127, 0.124, 0.125]
OVERFIT TEST PASSED


### 12a. Build the model

In [89]:
HP = make_hp()  # collect all the settings for this experiment
model, param_counts = build_model(HP)  # build the model and count its parameters

lorma | pool=attn | 152.7M parameters (152.7M used by the classifier), 8.81M trainable (5.77%)


### 12b. Train

In [ ]:
train_res = run_training(model, RUN_NAME, EPOCHS, LR, HEAD_LR)  # train the model and keep the training record

[p4_arch1_lorma_v2] 1500 optimizer steps per epoch (8 documents each), 3000 in total, time cap 300 min


p4_arch1_lorma_v2:   0%|          | 0/3000 [00:00<?, ?it/s]

[p4_arch1_lorma_v2] step 250/3000 | quick dev macro-F1 0.5489  <- best | 13.2 min
[p4_arch1_lorma_v2] step 500/3000 | quick dev macro-F1 0.5692  <- best | 26.3 min
[p4_arch1_lorma_v2] step 750/3000 | quick dev macro-F1 0.6260  <- best | 39.5 min
[p4_arch1_lorma_v2] step 1250/3000 | quick dev macro-F1 0.6786  <- best | 65.8 min
[p4_arch1_lorma_v2] step 1500/3000 | quick dev macro-F1 0.6967  <- best | 79.1 min
[p4_arch1_lorma_v2] step 1750/3000 | quick dev macro-F1 0.7275  <- best | 92.1 min
[p4_arch1_lorma_v2] step 2000/3000 | quick dev macro-F1 0.7279  <- best | 105.1 min
[p4_arch1_lorma_v2] step 2250/3000 | quick dev macro-F1 0.7469  <- best | 118.1 min
[p4_arch1_lorma_v2] step 2500/3000 | quick dev macro-F1 0.7520  <- best | 131.2 min
[p4_arch1_lorma_v2] step 2750/3000 | quick dev macro-F1 0.7510 | 144.2 min


### 12c. Validate: scores, threshold, calibration, false positives / negatives

In [ ]:
val_res = validate(model, RUN_NAME)  # evaluate on the dev set and print the full report

### 12d. Which part of the judgment matters? (reading only the first / last / first + last / all chunks)

In [ ]:
section_res = section_ablation(model, RUN_NAME)  # test reading only the first / last / first+last / all chunks
attn_res = attention_by_position(val_res['attn'])  # see which positions the model pays attention to

### 12e. Predict the test cases and write the submission

In [ ]:
test_probs, _ = predict_docs(model, test_docs)  # predict probabilities for the test cases
THRESHOLD = val_res['threshold'] if APPLY_TUNED_THRESHOLD else 0.5  # choose the cut-off to use
write_submission(test_docs, test_probs, THRESHOLD, OUTPUT_DIR)  # write prediction.json and submission.zip

### 12g. Blend with the TF-IDF reference model
The weight of the neural model and the decision threshold are chosen on the dev set; `prediction.json` / `submission.zip` are rewritten only if the blend beats the neural model alone on dev.

In [ ]:
y_dev = np.array([d['y'] for d in dev_docs])
if len(tfidf_dev) == len(dev_docs) and len(tfidf_test) == len(test_docs):
    neural_dev, neural_test = val_res['probs'], test_probs
    f_neural = tune_threshold(y_dev, neural_dev[:, 1])[1]
    cands = []
    for w in np.linspace(0, 1, 11):                                         # w = weight of the neural model
        thr_w, f_w = tune_threshold(y_dev, (w * neural_dev + (1 - w) * tfidf_dev)[:, 1])
        cands.append((f_w, w, thr_w))
    f_best, w_best, thr_best = max(cands)
    print(f'neural only (tuned threshold): {f_neural:.4f} | best blend: neural weight {w_best:.1f}, threshold {thr_best:.2f}, dev macro-F1 {f_best:.4f}')
    if f_best > f_neural + 1e-4:
        blend_test = w_best * neural_test + (1 - w_best) * tfidf_test
        write_submission(test_docs, blend_test, thr_best, OUTPUT_DIR)
        np.save(os.path.join(OUTPUT_DIR, 'blend_test_probs.npy'), blend_test)
        THRESHOLD = thr_best
    else:
        print('blend does not help on dev -> keeping the neural-only submission written in 12e')
else:
    print('TF-IDF predictions are not aligned with dev/test (smoke test?) -> skipped')

### 12f. Summary: size, memory, speed, scores

In [ ]:
summary_table([(f'{METHOD} / {SECTION_MODE}', param_counts, train_res, val_res)])  # print a one-row summary of this experiment

## 13. Save weights and results

In [ ]:
np.save(os.path.join(OUTPUT_DIR, 'dev_probs.npy'), val_res['probs']); np.save(os.path.join(OUTPUT_DIR, 'test_probs.npy'), test_probs)  # save dev and test probabilities (useful for ensembling)
json.dump(dict(dev_ids=[d['id'] for d in dev_docs], test_ids=[d['id'] for d in test_docs]), open(os.path.join(OUTPUT_DIR, 'ids.json'), 'w'))  # save the order of case ids
torch.save(dict(trainable=get_trainable_state(model), hp=HP, feat_dim=FEAT_DIM, chunk_len=CHUNK_LEN, section_mode=SECTION_MODE, threshold=THRESHOLD,
                phase1_ckpt=ckpt_path), os.path.join(OUTPUT_DIR, f'{RUN_NAME}_trainable.pt'))  # save only the trained weights plus the settings needed to reload them
drop = lambda v: {k: x for k, x in v.items() if k not in ('probs', 'attn')}  # helper: remove big items that should not go into the JSON
json.dump(dict(run=RUN_NAME, arch=ARCH_NAME, hp=HP, params=param_counts, train=train_res, validation=drop(val_res), section_ablation=section_res,
               attention_by_position=attn_res, threshold=THRESHOLD, class_counts=counts.tolist(), class_weights=cw.tolist()),
          open(os.path.join(OUTPUT_DIR, 'results.json'), 'w'), indent=1, default=str)  # save all results in results.json
print('saved:', sorted(os.listdir(OUTPUT_DIR)))  # list the files that were saved